[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/09_taylor_and_power_series/exercises.ipynb)

# Module 09 — Taylor and Power Series: Exercises

Forty fully solved problems in four tiers: **L0** (6), **L1** (12), **L2** (12), **L3** (10).

Each problem carries a **Statement**, an **Intuition**, a step-by-step **Solution**, a boxed
answer, a **Key takeaway**, and a code cell that recomputes the boxed answer independently.
Every numeric answer in this notebook was produced by the cell printed beneath it.

Notation follows [the register](../../docs/notation.md): the degree-$n$ Taylor polynomial is
$P_n$ with remainder $R_n$, and $T_k$ is reserved for Chebyshev polynomials. Theory, definitions
and proofs are in [first_principles.ipynb](first_principles.ipynb).

Run this cell first — every later code cell depends on it.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7.0, 4.0),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})
rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)

from math import factorial

print("environment ready; machine epsilon =", np.finfo(float).eps)

environment ready; machine epsilon = 2.220446049250313e-16


## L0 — Concept Checks

Six one-idea checks. Each is settled in a few lines and isolates a single distinction the rest of the module depends on.

### Problem L0.1 — Lagrange Remainder versus Local Asymptotics
**Source:** Apostol, *Calculus I*, Ch. 7

**Statement**

Explain the difference between the local asymptotic error $f(x) - P_n(x) = o((x-a)^n)$ and the Lagrange remainder formula $R_n(x) = \frac{f^{(n+1)}(c)}{(n+1)!}(x-a)^{n+1}$. When is Lagrange remainder strictly required?

**Intuition**
$o((x-a)^n)$ is a local statement about behavior as $x \to a$. It gives no numerical bound for a specific value $x = a + 0.1$. The Lagrange remainder provides an explicit numerical bound for the remainder over an entire interval, provided $f^{(n+1)}$ exists.

**Solution**
1. **Local Asymptotic Notation ($o((x-a)^n)$)**:  
   $f(x) = P_n(x) + o((x-a)^n)$ states that $\lim_{x\to a} \frac{R_n(x)}{(x-a)^n} = 0$. This requires only $f^{(n)}(a)$ to exist at the point $a$. It is qualitative and local.

2. **Lagrange Remainder Formula**:  
   $R_n(x) = \frac{f^{(n+1)}(c)}{(n+1)!}(x-a)^{n+1}$ for some $c \in (a, x)$.  
   This formula requires $f$ to be $(n+1)$-times differentiable on the entire interval $[a, x]$. It provides a quantitative, computable error bound:

$$
\lvert R_n(x) \rvert \le \frac{M_{n+1}}{(n+1)!} \lvert x-a \rvert^{n+1}, \quad \text{where } M_{n+1} = \sup_{t \in [a, x]} \lvert f^{(n+1)}(t) \rvert
$$

3. **When Lagrange Remainder is Required**:  
   - Computing numerical approximations with guaranteed precision (e.g. bounding $\sin(0.5)$ error to $\lt 10^{-6}$).
   - Proving global convergence of Taylor series ($R_n(x) \to 0$ as $n \to \infty$).

$$
\boxed{ \text{Local } o((x-a)^n) \text{ requires } f^{(n)}(a); \text{ Lagrange remainder requires } f^{(n+1)} \text{ on }[a,x] \text{ for explicit bounds.} }
$$

**Key takeaway**
Asymptotics describe limiting behavior near $a$; Lagrange remainders yield quantitative error bounds away from $a$.

In [2]:
# L0.1 — Peano is local and gives no bound; Lagrange gives a number at a fixed x.
f, a, n = np.exp, 0.0, 2
P2 = lambda x: 1 + x + x ** 2 / 2
print("Peano: R_n(x) / x^n -> 0 as x -> 0, but says nothing at a fixed x")
for x in (1e-1, 1e-2, 1e-3):
    print(f"  x = {x:.0e}: R_2(x)/x^2 = {(f(x)-P2(x))/x**2:.8f}")
print("\nLagrange: an explicit number at every x, from a bound on the third derivative")
for x in (0.5, 1.0, 2.0):
    M3 = np.exp(x)                              # sup of exp on [0, x]
    bound = M3 / factorial(3) * x ** 3
    true = abs(f(x) - P2(x))
    print(f"  x = {x:.1f}: |R_2| = {true:.8f} <= M_3 x^3 / 3! = {bound:.8f}"
          f"   (valid: {true <= bound})")
    assert true <= bound
print("\nthe Peano column shrinks but certifies nothing at x = 1; the Lagrange column does")

Peano: R_n(x) / x^n -> 0 as x -> 0, but says nothing at a fixed x
  x = 1e-01: R_2(x)/x^2 = 0.01709181
  x = 1e-02: R_2(x)/x^2 = 0.00167084
  x = 1e-03: R_2(x)/x^2 = 0.00016671

Lagrange: an explicit number at every x, from a bound on the third derivative
  x = 0.5: |R_2| = 0.02372127 <= M_3 x^3 / 3! = 0.03434836   (valid: True)
  x = 1.0: |R_2| = 0.21828183 <= M_3 x^3 / 3! = 0.45304697   (valid: True)
  x = 2.0: |R_2| = 2.38905610 <= M_3 x^3 / 3! = 9.85207480   (valid: True)

the Peano column shrinks but certifies nothing at x = 1; the Lagrange column does


### Problem L0.2 — Complex Singularities Set the Radius
**Source:** MIT 18.01 Calculus with Theory

**Statement**

Consider $f(x) = \frac{1}{1+x^2}$ for $x \in \mathbb{R}$.  
(a) Is $f(x)$ infinitely differentiable for all real $x$?  
(b) Find the Maclaurin series of $f(x)$ and its radius of convergence $R$.  
(c) Explain why $R$ is finite even though $f(x)$ has no singularities on the real line $\mathbb{R}$.

**Intuition**
Real calculus cannot explain why a smooth function like $\frac{1}{1+x^2}$ stops converging at $x = \pm 1$. The radius of convergence of a power series in $\mathbb{R}$ is determined by the distance from the center $a$ to the nearest singularity of $f(z)$ in the **complex plane** $\mathbb{C}$.

**Solution**
(a) Yes, $f(x) = (1+x^2)^{-1}$ is a rational function whose denominator $1+x^2 \ge 1 \gt 0$ for all $x \in \mathbb{R}$. Thus $f \in C^\infty(\mathbb{R})$.

(b) Using the geometric series formula $\frac{1}{1-u} = \sum_{n=0}^\infty u^n$ with $u = -x^2$:

$$
f(x) = \sum_{n=0}^\infty (-x^2)^n = \sum_{n=0}^\infty (-1)^n x^{2n} = 1 - x^2 + x^4 - x^6 + \dots
$$

The geometric series converges if and only if $\lvert -x^2 \rvert \lt 1 \iff \lvert x \rvert \lt 1$. Thus $R = 1$.

(c) In complex analysis, $f(z) = \frac{1}{1+z^2} = \frac{1}{(z-i)(z+i)}$ has simple poles at $z = \pm i$.  
The center of expansion is $z_0 = 0$. The distance from $0$ to the nearest complex singularity $z = \pm i$ is:

$$
\lvert 0 - (\pm i) \rvert = \sqrt{0^2 + 1^2} = 1
$$

The radius of convergence of a power series centered at $a$ equals the distance to the nearest singularity in $\mathbb{C}$.

$$
\boxed{ R = 1 \text{ because } f(z) = \frac{1}{1+z^2} \text{ has poles at } z = \pm i \text{ with distance } \lvert \pm i - 0 \rvert = 1 }
$$

**Key takeaway**
The radius of convergence of a real power series is governed by singularities in the complex domain.

In [3]:
# L0.2 — the Maclaurin series of 1/(1+x^2) and its radius.
N = 400
coeffs = np.array([(-1.0) ** (m // 2) if m % 2 == 0 else 0.0 for m in range(N)])
tail = np.arange(200, N, 2)
L = np.max(np.abs(coeffs[tail]) ** (1.0 / tail))
print(f"limsup |a_n|^(1/n) = {L:.12f}   ->  R = {1/L:.12f}")
print(f"distance from 0 to the poles at +/- i = {abs(1j - 0):.12f}")
for x in (0.5, 0.9, 0.99):
    s = float(np.sum(coeffs[:N] * x ** np.arange(N)))
    print(f"  x = {x:5.2f}: series = {s:.12f}   1/(1+x^2) = {1/(1+x*x):.12f}")
print(f"  x = 1.10: term 100 = {1.10 ** 100:.3e}  (terms blow up, series diverges)")
assert abs(1 / L - 1.0) < 1e-9
assert abs(float(np.sum(coeffs[:N] * 0.5 ** np.arange(N))) - 1 / 1.25) < 1e-12
print("verified: R = 1 = distance to the nearest complex singularity")

limsup |a_n|^(1/n) = 1.000000000000   ->  R = 1.000000000000
distance from 0 to the poles at +/- i = 1.000000000000
  x =  0.50: series = 0.800000000000   1/(1+x^2) = 0.800000000000
  x =  0.90: series = 0.552486187845   1/(1+x^2) = 0.552486187845
  x =  0.99: series = 0.495959520592   1/(1+x^2) = 0.505024998737
  x = 1.10: term 100 = 1.378e+04  (terms blow up, series diverges)
verified: R = 1 = distance to the nearest complex singularity


### Problem L0.3 — Big-O Asymptotic Simplification
**Source:** Bender & Orszag, *Advanced Mathematical Methods*, Ch. 1

**Statement**

Simplify the following asymptotic expression as $x \to 0$:

$$
f(x) = 3x^2 - 5x^4 + O(x^3) + 7x \cdot O(x^3) + O(x^5)
$$

**Intuition**
As $x \to 0$, lower powers of $x$ dominate larger powers. Any term $x^k$ with $k \ge 3$ is absorbed into $O(x^3)$.

**Solution**
Apply the algebraic rules of Big-O notation near $x=0$:
1. $-5x^4 = O(x^4) \subset O(x^3)$ since $4 \ge 3$.
2. $7x \cdot O(x^3) = O(x^{1+3}) = O(x^4) \subset O(x^3)$.
3. $O(x^5) \subset O(x^3)$.

Combining error terms:

$$
O(x^3) + O(x^4) + O(x^4) + O(x^5) = O(x^3)
$$

Thus:

$$
f(x) = 3x^2 + O(x^3)
$$

$$
\boxed{ f(x) = 3x^2 + O(x^3) }
$$

**Key takeaway**
Near origin $x=0$, the smallest power $x^m$ bounds all higher powers $x^n$ ($n \ge m$), absorbing them into $O(x^m)$.

In [4]:
# L0.3 — the claim f(x) = 3x^2 + O(x^3) means |f(x) - 3x^2| / x^3 stays bounded.
def f_sample(x, c3=1.0, c5=2.0):
    """3x^2 - 5x^4 with concrete admissible choices for the three O-terms."""
    return 3 * x ** 2 - 5 * x ** 4 + c3 * x ** 3 + 7 * x * (c5 * x ** 3) + c5 * x ** 5

xs = np.array([1e-1, 1e-2, 1e-3, 1e-4])
ratio = np.abs(f_sample(xs) - 3 * xs ** 2) / xs ** 3
print("x            |f(x) - 3x^2| / x^3")
for x, r in zip(xs, ratio):
    print(f"{x:.0e}        {r:.6f}")
print(f"\nratio stays bounded (max = {ratio.max():.4f}) -> the x^3 exponent is right")
# and 3x^2 + O(x^2) would be a weaker, non-sharp statement; O(x^4) is false:
print(f"|f - 3x^2| / x^4 at x = 1e-4 : {abs(f_sample(1e-4) - 3e-8) / 1e-16:.3e}  (unbounded)")
assert ratio.max() < 2.0
assert abs(f_sample(1e-4) - 3e-8) / 1e-16 > 1e3

x            |f(x) - 3x^2| / x^3
1e-01        1.920000
1e-02        1.090200
1e-03        1.009002
1e-04        1.000900

ratio stays bounded (max = 1.9200) -> the x^3 exponent is right
|f - 3x^2| / x^4 at x = 1e-4 : 1.001e+04  (unbounded)


### Problem L0.4 — Cauchy Product of Two Geometric Series
**Source:** Apostol, *Calculus I*, Ch. 9

**Statement**

Let $A(x) = \sum_{n=0}^\infty x^n$ and $B(x) = \sum_{n=0}^\infty x^n$ for $\lvert x \rvert \lt 1$. Compute their Cauchy product $C(x) = A(x)B(x)$ and determine its sum in closed form.

**Intuition**
The Cauchy product corresponds to discrete convolution of coefficients. Since $A(x) = B(x) = \frac{1}{1-x}$, their product is $\frac{1}{(1-x)^2}$, which is the derivative of $\frac{1}{1-x}$.

**Solution**
The Cauchy product formula states that $\left(\sum a_n x^n\right)\left(\sum b_n x^n\right) = \sum c_n x^n$ where $c_n = \sum_{k=0}^n a_k b_{n-k}$.  
Here $a_k = 1$ and $b_{n-k} = 1$ for all $k$.

Compute $c_n$:

$$
c_n = \sum_{k=0}^n (1)(1) = n + 1
$$

Thus:

$$
C(x) = \sum_{n=0}^\infty (n+1) x^n = 1 + 2x + 3x^2 + 4x^3 + \dots
$$

Closed form evaluation:

$$
A(x) = \frac{1}{1-x} \implies A(x) B(x) = \frac{1}{(1-x)^2}
$$

Notice that $\frac{d}{dx} \left( \frac{1}{1-x} \right) = \frac{d}{dx} \sum_{n=0}^\infty x^n = \sum_{n=1}^\infty n x^{n-1} = \sum_{k=0}^\infty (k+1) x^k$, matching $C(x)$.

$$
\boxed{ C(x) = \sum_{n=0}^\infty (n+1)x^n = \frac{1}{(1-x)^2} \quad \text{for } \lvert x \rvert \lt 1 }
$$

**Key takeaway**
Cauchy multiplication of power series computes discrete coefficient convolutions, matching function multiplication in their shared convergence domain.

In [5]:
# L0.4 — the Cauchy product coefficients c_n = n+1, and the closed form 1/(1-x)^2.
N = 400
a = np.ones(N)
c = np.convolve(a, a)[:N]             # discrete convolution = Cauchy product
print("c_n for n = 0..7 :", c[:8].astype(int), " expected n+1 :", np.arange(1, 9))
assert np.allclose(c, np.arange(1, N + 1))
for x in (0.25, 0.5, 0.8):
    s = float(np.sum(c * x ** np.arange(N)))
    print(f"  x = {x:4.2f}: sum (n+1)x^n = {s:.12f}   1/(1-x)^2 = {1/(1-x)**2:.12f}")
    assert abs(s - 1 / (1 - x) ** 2) < 1e-9
print("verified: C(x) = 1/(1-x)^2 on |x| < 1")

c_n for n = 0..7 : [1 2 3 4 5 6 7 8]  expected n+1 : [1 2 3 4 5 6 7 8]
  x = 0.25: sum (n+1)x^n = 1.777777777778   1/(1-x)^2 = 1.777777777778
  x = 0.50: sum (n+1)x^n = 4.000000000000   1/(1-x)^2 = 4.000000000000
  x = 0.80: sum (n+1)x^n = 25.000000000000   1/(1-x)^2 = 25.000000000000
verified: C(x) = 1/(1-x)^2 on |x| < 1


### Problem L0.5 — Euler's Formula as a Rotation
**Source:** Spivak, *Calculus*, Ch. 26

**Statement**

Using Euler's formula $e^{i\theta} = \cos\theta + i\sin\theta$, prove that multiplication of a complex number $z = r e^{i\phi}$ by $e^{i\theta}$ corresponds geometrically to a counterclockwise rotation of $z$ by angle $\theta$ in the complex plane without changing its magnitude.

**Intuition**
Complex multiplication adds arguments (angles) and multiplies moduli (lengths). Since $\lvert e^{i\theta} \rvert = 1$, multiplying by $e^{i\theta}$ preserves distance to origin while rotating the vector by angle $\theta$.

**Solution**
Let $z = r(\cos\phi + i\sin\phi) = r e^{i\phi}$.  
Multiply $z$ by $w = e^{i\theta} = \cos\theta + i\sin\theta$:

$$
\begin{aligned} z \cdot w &= (r e^{i\phi})(e^{i\theta}) = r e^{i(\phi + \theta)} \\ &= r \left[ \cos(\phi + \theta) + i \sin(\phi + \theta) \right] \end{aligned}
$$

Modulus check:

$$
\lvert z \cdot w \rvert = \lvert r e^{i(\phi+\theta)} \rvert = r \lvert e^{i(\phi+\theta)} \rvert = r \sqrt{\cos^2(\phi+\theta) + \sin^2(\phi+\theta)} = r = \lvert z \rvert
$$

Argument check:

$$
\arg(z \cdot w) = \phi + \theta = \arg(z) + \theta
$$

Thus, the length remains $r$, and the direction is rotated counterclockwise by angle $\theta$.

$$
\boxed{ \lvert z \cdot e^{i\theta} \rvert = \lvert z \rvert, \quad \arg(z \cdot e^{i\theta}) = \arg(z) + \theta }
$$

**Key takeaway**
The operator $e^{i\theta}$ acts as a pure rotation matrix in $\mathbb{R}^2$:

$$
\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}
$$

In [6]:
# L0.5 — multiplication by e^{i theta} preserves modulus and adds theta to the argument.
z = 2.0 * np.exp(1j * 0.4)
for theta in (0.3, 1.0, 2.5):
    w = z * np.exp(1j * theta)
    print(f"theta = {theta:4.2f}:  |z| = {abs(z):.12f} -> |zw| = {abs(w):.12f}"
          f"   arg: {np.angle(z):.6f} -> {np.angle(w):.6f}  (+{theta:.2f})")
    assert abs(abs(w) - abs(z)) < 1e-14
    assert abs((np.angle(w) - np.angle(z)) % (2 * np.pi) - theta) < 1e-12
# the same map written as the 2x2 rotation matrix
theta = 1.0
R = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
v = np.array([z.real, z.imag])
w = z * np.exp(1j * theta)
print("\nrotation matrix image :", R @ v, "   complex product :", np.array([w.real, w.imag]))
assert np.allclose(R @ v, [w.real, w.imag])

theta = 0.30:  |z| = 2.000000000000 -> |zw| = 2.000000000000   arg: 0.400000 -> 0.700000  (+0.30)
theta = 1.00:  |z| = 2.000000000000 -> |zw| = 2.000000000000   arg: 0.400000 -> 1.400000  (+1.00)
theta = 2.50:  |z| = 2.000000000000 -> |zw| = 2.000000000000   arg: 0.400000 -> 2.900000  (+2.50)

rotation matrix image : [0.3399 1.9709]    complex product : [0.3399 1.9709]


### Problem L0.6 — A Smooth Function That Is Not Analytic
**Source:** Spivak, *Calculus*, Ch. 19

**Statement**

Let:

$$
f(x) = \begin{cases} e^{-1/x^2} & \text{if } x \neq 0 \\ 0 & \text{if } x = 0 \end{cases}
$$

(a) Show that $f^{(n)}(0) = 0$ for all $n \ge 0$.  
(b) What is the Maclaurin series of $f(x)$? Does it equal $f(x)$ away from $x=0$?

**Intuition**
Exponential decay $e^{-1/x^2}$ vanishes faster than any polynomial $x^k$ as $x \to 0$. Hence, every derivative at $x=0$ evaluates to 0.

**Solution**
(a) For $x \neq 0$, by induction $f^{(n)}(x) = P_n(\frac{1}{x}) e^{-1/x^2}$ where $P_n$ is a polynomial.  
For $x \to 0$, substitute $u = 1/x^2 \to \infty$:

$$
\lim_{x\to 0} \frac{e^{-1/x^2}}{x^k} = \lim_{u\to\infty} u^{\frac{k}{2}} e^{-u} = 0 \quad (\text{by L'Hôpital's Rule})
$$

By definition of the derivative at 0:

$$
f'(0) = \lim_{x\to 0} \frac{e^{-1/x^2} - 0}{x} = 0
$$

Inductively, $f^{(n)}(0) = 0$ for all $n \ge 0$.

(b) The Maclaurin series of $f(x)$ is:

$$
\sum_{n=0}^\infty \frac{f^{(n)}(0)}{n!} x^n = \sum_{n=0}^\infty 0 \cdot x^n = 0
$$

However, for any $x \neq 0$, $f(x) = e^{-1/x^2} \gt 0 \neq 0$.  
Thus, the Maclaurin series converges for all $x \in \mathbb{R}$, but equals $f(x)$ **only at $x=0$**.

$$
\boxed{ f^{(n)}(0) = 0 \, \forall n \implies M(x) = 0 \neq f(x) \text{ for } x \neq 0 }
$$

**Key takeaway**
$C^\infty$ smoothness does NOT imply analyticity! A function can be infinitely differentiable without equaling its Taylor series.

In [7]:
# L0.6 — every difference quotient defining f^(n)(0) collapses to zero.
def f_flat(x):
    x = np.asarray(x, dtype=float)
    out = np.zeros_like(x)
    nz = x != 0
    out[nz] = np.exp(-1.0 / x[nz] ** 2)
    return out

hs = np.array([0.5, 0.2, 0.1, 0.05])
print("  h        f(h)          f(h)/h^10       f(h)/h^30")
for h in hs:
    v = float(f_flat(h))
    print(f"{h:5.2f}   {v:.4e}   {v/h**10:.4e}   {v/h**30:.4e}")
print("\nMaclaurin series value at every x is 0, so R_n(x) = f(x) for every n:")
for x in (0.5, 0.3, 0.1):
    print(f"  x = {x}:  f(x) = {float(f_flat(x)):.6e}   series = 0   |R_n| = {float(f_flat(x)):.6e}")
assert float(f_flat(0.05)) / 0.05 ** 30 < 1e-100
assert float(f_flat(0.5)) > 0
print("\nf vanishes faster than every power of x, yet is strictly positive off the origin")

  h        f(h)          f(h)/h^10       f(h)/h^30
 0.50   1.8316e-02   1.8755e+01   1.9666e+07
 0.20   1.3888e-11   1.3562e-04   1.2934e+10
 0.10   3.7201e-44   3.7201e-34   3.7201e-14
 0.05   1.9152e-174   1.9611e-161   2.0564e-135

Maclaurin series value at every x is 0, so R_n(x) = f(x) for every n:
  x = 0.5:  f(x) = 1.831564e-02   series = 0   |R_n| = 1.831564e-02
  x = 0.3:  f(x) = 1.494534e-05   series = 0   |R_n| = 1.494534e-05
  x = 0.1:  f(x) = 3.720076e-44   series = 0   |R_n| = 3.720076e-44

f vanishes faster than every power of x, yet is strictly positive off the origin


## L1 — Foundations

Twelve core computations: remainder bounds, radii of convergence, term-by-term calculus and endpoint sums.

### Problem L1.1 — Contact Order of the Taylor Polynomial
**Source:** Spivak, *Calculus*, Ch. 19

**Statement**

Let $f(x)$ be $n$-times differentiable at $x=a$, and let $P_n(x) = \sum_{k=0}^n \frac{f^{(k)}(a)}{k!}(x-a)^k$. Prove directly from the definition of Taylor polynomials that:

$$
\lim_{x\to a} \frac{f(x) - P_n(x)}{(x-a)^n} = 0
$$

**Intuition**
The Taylor polynomial $P_n(x)$ is designed to match all derivatives of $f(x)$ up to order $n$ at $x=a$. Therefore, the difference $f(x) - P_n(x)$ and its first $n-1$ derivatives all vanish at $x=a$. Applying L'Hôpital's Rule $n-1$ times will reduce the limit to the definition of the derivative $f^{(n)}(a)$.

**Solution**
Define $E(x) = f(x) - P_n(x)$.  
Note that for any $k \in \{0, 1, \dots, n-1\}$:

$$
P_n^{(k)}(a) = f^{(k)}(a) \implies E^{(k)}(a) = f^{(k)}(a) - P_n^{(k)}(a) = 0
$$

Also for $k=n-1$:

$$
P_n^{(n-1)}(x) = f^{(n-1)}(a) + f^{(n)}(a)(x-a)
$$

$$
E^{(n-1)}(x) = f^{(n-1)}(x) - f^{(n-1)}(a) - f^{(n)}(a)(x-a)
$$

Consider the limit:

$$
L = \lim_{x\to a} \frac{E(x)}{(x-a)^n}
$$

Since $E(a) = E'(a) = \dots = E^{(n-1)}(a) = 0$ and the denominator $(x-a)^n$ and its derivatives up to order $n-1$ vanish at $x=a$, we apply L'Hôpital's Rule $n-1$ times sequentially:

$$
\begin{aligned} L &= \lim_{x\to a} \frac{E'(x)}{n(x-a)^{n-1}} = \dots = \lim_{x\to a} \frac{E^{(n-1)}(x)}{n! (x-a)} \\ &= \lim_{x\to a} \frac{f^{(n-1)}(x) - f^{(n-1)}(a) - f^{(n)}(a)(x-a)}{n! (x-a)} \\ &= \frac{1}{n!} \left( \lim_{x\to a} \frac{f^{(n-1)}(x) - f^{(n-1)}(a)}{x-a} - f^{(n)}(a) \right) \end{aligned}
$$

By definition of the derivative $f^{(n)}(a) = \lim_{x\to a} \frac{f^{(n-1)}(x) - f^{(n-1)}(a)}{x-a}$, the term inside parentheses is $f^{(n)}(a) - f^{(n)}(a) = 0$.  
Thus, $L = \frac{1}{n!}(0) = 0$.

$$
\boxed{ \lim_{x\to a} \frac{f(x) - P_n(x)}{(x-a)^n} = 0 }
$$

**Key takeaway**
Matching $n$ derivatives guarantees that the error term vanishes strictly faster than $(x-a)^n$. This defines contact order $n$.

In [8]:
# L1.1 — measure the contact order: |f(x) - P_n(x)| / |x - a|^n must tend to 0.
def taylor_poly(derivs, a, n, x):
    return sum(derivs[k] * (x - a) ** k / factorial(k) for k in range(n + 1))

a = 0.3
f = np.sin
derivs = [np.sin(a + k * np.pi / 2) for k in range(10)]
hs = np.array([1e-1, 1e-2, 1e-3, 1e-4])
print("  n      h        |f - P_n| / h^n        |f - P_n| / h^(n+1)")
for n in (1, 2, 3):
    for h in hs:
        err = abs(f(a + h) - taylor_poly(derivs, a, n, a + h))
        print(f"{n:3d}   {h:.0e}   {err/h**n:20.10e}   {err/h**(n+1):18.6e}")
    print()
for n in (1, 2, 3):
    r = [abs(f(a + h) - taylor_poly(derivs, a, n, a + h)) / h ** n for h in hs]
    assert r[-1] < r[0] / 100
print("the ratio / h^n -> 0 while / h^(n+1) stays bounded: contact of order exactly n")

  n      h        |f - P_n| / h^n        |f - P_n| / h^(n+1)
  1   1e-01       1.6355132652e-02         1.635513e-01
  1   1e-02       1.4935109152e-03         1.493511e-01
  1   1e-03       1.4791931374e-04         1.479193e-01
  1   1e-04       1.4777602475e-05         1.477760e-01

  2   1e-01       1.5791223194e-02         1.579122e-01
  2   1e-02       1.5909881906e-03         1.590988e-01
  2   1e-03       1.5921042262e-04         1.592104e-01
  2   1e-04       1.5920598173e-05         1.592060e-01

  3   1e-01       1.3105162446e-03         1.310516e-02
  3   1e-02       1.2392914472e-04         1.239291e-02
  3   1e-03       1.2323475573e-05         1.232348e-02
  3   1e-04       0.0000000000e+00         0.000000e+00

the ratio / h^n -> 0 while / h^(n+1) stays bounded: contact of order exactly n


### Problem L1.2 — Binomial Polynomial and Lagrange Error Bound
**Source:** Stewart, *Calculus*, Ch. 11

**Statement**

Compute the degree-3 Maclaurin polynomial $P_3(x)$ for $f(x) = \sqrt{1+x}$ and bound the error $\lvert R_3(x) \rvert$ for $x \in [0, 0.2]$.

**Intuition**
Apply the binomial series formula $(1+x)^\alpha = 1 + \alpha x + \frac{\alpha(\alpha-1)}{2} x^2 + \dots$ with $\alpha = \frac{1}{2}$. The 4th derivative provides the Lagrange error bound.

**Solution**
Compute derivatives of $f(x) = (1+x)^{\frac{1}{2}}$:

$$
\begin{aligned} f(0) &= 1 \\ f'(x) &= \frac{1}{2}(1+x)^{-\frac{1}{2}} \implies f'(0) = \frac{1}{2} \\ f''(x) &= -\frac{1}{4}(1+x)^{-\frac{3}{2}} \implies f''(0) = -\frac{1}{4} \\ f'''(x) &= \frac{3}{8}(1+x)^{-\frac{5}{2}} \implies f'''(0) = \frac{3}{8} \\ f^{(4)}(x) &= -\frac{15}{16}(1+x)^{-\frac{7}{2}} \end{aligned}
$$

Construct $P_3(x)$:

$$
P_3(x) = 1 + \frac{1}{2}x - \frac{1}{8}x^2 + \frac{1}{16}x^3
$$

Lagrange remainder $R_3(x)$ for $x \in [0, 0.2]$:

$$
R_3(x) = \frac{f^{(4)}(c)}{4!} x^4 = \frac{-\frac{15}{16} (1+c)^{-\frac{7}{2}}}{24} x^4 = -\frac{5}{128(1+c)^{\frac{7}{2}}} x^4 \quad \text{for } c \in (0, x)
$$

Since $c \gt 0$, $(1+c)^{-\frac{7}{2}} \lt 1$. Maximize $\lvert R_3(x) \rvert$ on $[0, 0.2]$ at $x = 0.2$:

$$
\lvert R_3(x) \rvert \le \frac{5}{128} (0.2)^4 = \frac{5}{128} \times 0.0016 = \frac{0.008}{128} = 6.25 \times 10^{-5}
$$

$$
\boxed{ P_3(x) = 1 + \frac{x}{2} - \frac{x^2}{8} + \frac{x^3}{16}, \quad \lvert R_3(x) \rvert \le 6.25 \times 10^{-5} }
$$

**Key takeaway**
For alternating signs in derivatives, evaluating remainder bounds at interval endpoints gives sharp numerical guarantees.

In [9]:
# L1.2 — P_3 for sqrt(1+x) and the Lagrange bound 6.25e-5 on [0, 0.2].
xs = np.linspace(0.0, 0.2, 2001)
P3 = 1 + xs / 2 - xs ** 2 / 8 + xs ** 3 / 16
err = np.abs(np.sqrt(1 + xs) - P3)
bound = 5 / 128 * 0.2 ** 4
print(f"claimed Lagrange bound        = {bound:.6e}")
print(f"largest true error on [0,0.2] = {err.max():.6e}   at x = {xs[err.argmax()]:.4f}")
print(f"bound / true error            = {bound/err.max():.4f}")
assert abs(bound - 6.25e-5) < 1e-12
assert err.max() < bound
print("\nverified: 5/128 * (0.2)^4 = 6.25e-5 is a valid upper bound")

claimed Lagrange bound        = 6.250000e-05
largest true error on [0,0.2] = 5.488499e-05   at x = 0.2000
bound / true error            = 1.1387

verified: 5/128 * (0.2)^4 = 6.25e-5 is a valid upper bound


### Problem L1.3 — Certified Precision for $\sin(0.5)$
**Source:** Spivak, *Calculus*, Ch. 19

**Statement**

Estimate $\sin(0.5)$ using the degree-5 Maclaurin polynomial $P_5(x)$. Prove that the error is strictly bounded by $3 \times 10^{-6}$.

**Intuition**
The Maclaurin series for $\sin x$ contains only odd powers. $P_5(x) = x - \frac{x^3}{6} + \frac{x^5}{120}$. By the alternating series estimation theorem (or Lagrange remainder), the error is bounded by the magnitude of the next term $\frac{x^7}{5040}$.

**Solution**
For $f(x) = \sin x$, the Maclaurin expansion is:

$$
P_5(x) = x - \frac{x^3}{3!} + \frac{x^5}{5!} = x - \frac{x^3}{6} + \frac{x^5}{120}
$$

Evaluate at $x = 0.5 = \frac{1}{2}$:

$$
P_5(0.5) = \frac{1}{2} - \frac{1}{48} + \frac{1}{3840} = \frac{1920 - 80 + 1}{3840} = \frac{1841}{3840} \approx 0.479427083
$$

Lagrange Remainder $R_5(0.5)$:

$$
R_5(x) = \frac{f^{(6)}(c)}{6!} x^6 = \frac{-\sin(c)}{720} x^6 \implies \lvert R_5(0.5) \rvert \le \frac{\sin(0.5)}{720} (0.5)^6
$$

Since $\sin(0.5) \lt 0.5$:

$$
\lvert R_5(0.5) \rvert \lt \frac{0.5}{720 \cdot 64} = \frac{1}{92160} \approx 1.085 \times 10^{-5}
$$

Alternatively, using $P_6(x) = P_5(x)$ (since $a_6 = 0$), the remainder $R_6(x) = \frac{f^{(7)}(c)}{7!} x^7 = \frac{-\cos(c)}{5040} x^7$:

$$
\lvert R_6(0.5) \rvert \le \frac{1}{5040} (0.5)^7 = \frac{1}{5040 \cdot 128} = \frac{1}{645120} \approx 1.55 \times 10^{-6} \lt 3 \times 10^{-6}
$$

$$
\boxed{ \sin(0.5) \approx \frac{1841}{3840}, \quad \lvert R \rvert \le \frac{1}{645120} \approx 1.55 \times 10^{-6} }
$$

**Key takeaway**
Since $a_6 = 0$, $P_5(x) = P_6(x)$, allowing us to use the degree-7 derivative for a tighter error bound.

In [10]:
# L1.3 — P_5(0.5) = 1841/3840 and the certified error bound 1/645120.
P5 = 0.5 - 0.5 ** 3 / 6 + 0.5 ** 5 / 120
exact_frac = 1841 / 3840
true = np.sin(0.5)
err = abs(P5 - true)
bound = 1 / 645120
print(f"P_5(0.5)      = {P5:.12f}")
print(f"1841/3840     = {exact_frac:.12f}")
print(f"sin(0.5)      = {true:.12f}")
print(f"true error    = {err:.6e}")
print(f"claimed bound = {bound:.6e}   (= 0.5^7 / 7!)")
print(f"3e-6 target   : bound < 3e-6 is {bound < 3e-6}")
assert abs(P5 - exact_frac) < 1e-15
assert err < bound < 3e-6
print("\nverified: the degree-7 bound certifies the answer to better than 3e-6")

P_5(0.5)      = 0.479427083333
1841/3840     = 0.479427083333
sin(0.5)      = 0.479425538604
true error    = 1.544729e-06
claimed bound = 1.550099e-06   (= 0.5^7 / 7!)
3e-6 target   : bound < 3e-6 is True

verified: the degree-7 bound certifies the answer to better than 3e-6


### Problem L1.4 — Explicit Integral Remainder for $\ln(1+x)$
**Source:** Apostol, *Calculus I*, Ch. 7

**Statement**

Compute the Integral Remainder $R_2(x)$ for $f(x) = \ln(1+x)$ centered at $a=0$. Use it to derive the bound $\lvert R_2(x) \rvert \le \frac{\lvert x \rvert^3}{3(1-\lvert x \rvert)^3}$ for $x \in (-1, 0)$.

**Intuition**
Apply the integral remainder formula $R_n(x) = \frac{1}{n!} \int_0^x f^{(n+1)}(t)(x-t)^n dt$ for $n=2$.

**Solution**
Derivatives of $f(t) = \ln(1+t)$:

$$
f'(t) = \frac{1}{1+t}, \quad f''(t) = -\frac{1}{(1+t)^2}, \quad f'''(t) = \frac{2}{(1+t)^3}
$$

Substitute into Integral Remainder formula for $n=2$:

$$
R_2(x) = \frac{1}{2!} \int_{0}^{x} \frac{2}{(1+t)^3} (x-t)^2 \, dt = \int_{0}^{x} \frac{(x-t)^2}{(1+t)^3} \, dt
$$

For $x \in (-1, 0)$, write $t = -s$ where $s \in [0, \lvert x \rvert]$. Since $t \in [x, 0]$, $1+t \ge 1+x = 1-\lvert x \rvert \gt 0$.  
Thus $\frac{1}{(1+t)^3} \le \frac{1}{(1-\lvert x \rvert)^3}$.

Bound the integral:

$$
\lvert R_2(x) \rvert \le \frac{1}{(1-\lvert x \rvert)^3} \left\vert \int_{0}^{x} (x-t)^2 \, dt \right\vert = \frac{1}{(1-\lvert x \rvert)^3} \left[ \frac{(x-t)^3}{-3} \right]_0^x = \frac{\lvert x \rvert^3}{3(1-\lvert x \rvert)^3}
$$

$$
\boxed{ R_2(x) = \int_0^x \frac{(x-t)^2}{(1+t)^3} \, dt, \quad \lvert R_2(x) \rvert \le \frac{\lvert x \rvert^3}{3(1-\lvert x \rvert)^3} \text{ for } x \in (-1, 0) }
$$

**Key takeaway**
The integral remainder allows bounding denominators by their minimum value over the integration domain.

In [11]:
# L1.4 — the integral remainder for ln(1+x) at n = 2, and the bound |x|^3 / (3(1-|x|)^3).
from scipy.integrate import quad

print("   x        R_2 exact        integral form        bound |x|^3/(3(1-|x|)^3)")
for x in (-0.2, -0.4, -0.6, -0.8):
    R2 = np.log1p(x) - (x - x ** 2 / 2)
    R2_int = quad(lambda t: (x - t) ** 2 / (1 + t) ** 3, 0.0, x)[0]
    bnd = abs(x) ** 3 / (3 * (1 - abs(x)) ** 3)
    print(f"{x:6.2f}   {R2:14.8e}   {R2_int:16.8e}   {bnd:20.8e}")
    assert abs(R2 - R2_int) < 1e-12
    assert abs(R2) <= bnd
print("\nverified: the integral form is exact, and the stated bound holds on (-1, 0)")

   x        R_2 exact        integral form        bound |x|^3/(3(1-|x|)^3)
 -0.20   -3.14355131e-03    -3.14355131e-03         5.20833333e-03
 -0.40   -3.08256238e-02    -3.08256238e-02         9.87654321e-02
 -0.60   -1.36290732e-01    -1.36290732e-01         1.12500000e+00
 -0.80   -4.89437912e-01    -4.89437912e-01         2.13333333e+01

verified: the integral form is exact, and the stated bound holds on (-1, 0)


### Problem L1.5 — Radius by the Ratio Test
**Source:** Demidovich, *Problems in Analysis*, No. 2530

**Statement**

Find the radius of convergence $R$ and the exact interval of convergence for the power series:

$$
\sum_{n=1}^\infty \frac{(n!)^2}{(2n)!} x^n
$$

**Intuition**
The ratio of factorials $\frac{a_{n+1}}{a_n}$ simplifies rapidly. Apply D'Alembert's ratio test to extract $R$.

**Solution**
Let $a_n = \frac{(n!)^2}{(2n)!}$. Compute the ratio $\left\vert\frac{a_{n+1}}{a_n}\right\vert$:

$$
\begin{aligned} \left\vert \frac{a_{n+1}}{a_n} \right\vert &= \frac{((n+1)!)^2}{(2n+2)!} \cdot \frac{(2n)!}{(n!)^2} \\ &= \frac{(n+1)^2}{(2n+2)(2n+1)} = \frac{(n+1)^2}{2(n+1)(2n+1)} = \frac{n+1}{2(2n+1)} \end{aligned}
$$

Take the limit as $n \to \infty$:

$$
L = \lim_{n\to\infty} \left\vert \frac{a_{n+1}}{a_n} \right\vert = \lim_{n\to\infty} \frac{n+1}{4n+2} = \frac{1}{4}
$$

By D'Alembert's ratio test, $R = \frac{1}{L} = 4$.

Check Endpoints $x = \pm 4$:
For $x = 4$:

$$
u_n = a_n 4^n = \frac{(n!)^2 4^n}{(2n)!}
$$

Using Stirling's approximation $n! \sim \sqrt{2\pi n} \left(\frac{n}{e}\right)^n$:

$$
u_n \sim \frac{2\pi n \left(\frac{n}{e}\right)^{2n} 4^n}{\sqrt{4\pi n} \left(\frac{2n}{e}\right)^{2n}} = \sqrt{\pi n} \frac{n^{2n} 4^n}{2^{2n} n^{2n}} = \sqrt{\pi n}
$$

Since $\lim_{n\to\infty} u_n = \infty \neq 0$, the series diverges at $x = 4$ and $x = -4$.

$$
\boxed{ R = 4, \quad \text{Interval of Convergence: } (-4, 4) }
$$

**Key takeaway**
Stirling's approximation is the standard tool for testing endpoint convergence of factorial power series.

In [12]:
# L1.5 — R = 4 for sum (n!)^2 / (2n)! x^n, and divergence at both endpoints.
from math import lgamma

def log_a(n):
    return 2 * lgamma(n + 1) - lgamma(2 * n + 1)

ns = np.arange(1, 400)
la = np.array([log_a(int(n)) for n in ns])
ratio = np.exp(la[1:] - la[:-1])
root = np.exp(la / ns)
print(f"a_(n+1)/a_n at n = 399 : {ratio[-1]:.10f}   (limit 1/4 = 0.25)")
print(f"|a_n|^(1/n)  at n = 399 : {root[-1]:.10f}   (limit 1/4)")
print(f"R = 1 / limit           : {1/ratio[-1]:.6f}   (claimed 4)")
terms4 = np.exp(la + ns * np.log(4.0))
print(f"\nterms at x = 4, n = 100, 200, 399 : "
      f"{terms4[99]:.4f}, {terms4[199]:.4f}, {terms4[-1]:.4f}")
print(f"Stirling prediction sqrt(pi n)   : "
      f"{np.sqrt(np.pi*100):.4f}, {np.sqrt(np.pi*200):.4f}, {np.sqrt(np.pi*399):.4f}")
assert abs(1 / ratio[-1] - 4.0) < 0.02
assert abs(terms4[-1] / np.sqrt(np.pi * 399) - 1.0) < 0.01
print("\nverified: R = 4, and at x = +/- 4 the terms grow like sqrt(pi n), so both endpoints diverge")

a_(n+1)/a_n at n = 399 : 0.2503136763   (limit 1/4 = 0.25)
|a_n|^(1/n)  at n = 399 : 0.2522450828   (limit 1/4)
R = 1 / limit           : 3.994987   (claimed 4)

terms at x = 4, n = 100, 200, 399 : 17.7467, 25.0820, 35.4158
Stirling prediction sqrt(pi n)   : 17.7245, 25.0663, 35.4047

verified: R = 4, and at x = +/- 4 the terms grow like sqrt(pi n), so both endpoints diverge


### Problem L1.6 — Radius by the Root Test
**Source:** Kaczor & Nowak, *Problems in Mathematical Analysis III*

**Statement**

Determine the radius of convergence of the power series:

$$
\sum_{n=1}^\infty \left( 1 + \frac{1}{n} \right)^{n^2} x^n
$$

**Intuition**
The exponent $n^2$ strongly suggests applying the Cauchy Root Test $\sqrt[n]{\lvert a_n \rvert}$.

**Solution**
Let $a_n = \left( 1 + \frac{1}{n} \right)^{n^2}$.  
Apply the Cauchy Root Test:

$$
\sqrt[n]{\lvert a_n \rvert} = \left[ \left( 1 + \frac{1}{n} \right)^{n^2} \right]^{\frac{1}{n}} = \left( 1 + \frac{1}{n} \right)^n
$$

Take the limit as $n \to \infty$:

$$
L = \lim_{n\to\infty} \sqrt[n]{\lvert a_n \rvert} = \lim_{n\to\infty} \left( 1 + \frac{1}{n} \right)^n = e
$$

By the Cauchy-Hadamard Theorem:

$$
R = \frac{1}{L} = \frac{1}{e} = e^{-1}
$$

$$
\boxed{ R = \frac{1}{e} }
$$

**Key takeaway**
When $a_n$ involves $n$-th powers of $n$-dependent terms, the Root Test avoids complex ratio simplifications.

In [13]:
# L1.6 — R = 1/e for sum (1 + 1/n)^(n^2) x^n by the root test.
ns = np.arange(1, 20001)
root = (1.0 + 1.0 / ns) ** ns          # = |a_n|^(1/n)
print(f"|a_n|^(1/n) at n = 10, 1e3, 2e4 : {root[9]:.10f}, {root[999]:.10f}, {root[-1]:.10f}")
print(f"e                                = {np.e:.10f}")
print(f"R = 1 / limsup                   = {1/root[-1]:.10f}   (1/e = {1/np.e:.10f})")
assert abs(root[-1] - np.e) < 1e-3
assert abs(1 / root[-1] - 1 / np.e) < 1e-3
# terms at |x| slightly inside and slightly outside 1/e
for x, tag in [(0.9 / np.e, "inside"), (1.1 / np.e, "outside")]:
    with np.errstate(over="ignore"):
        t = root[:200] ** np.arange(1, 201) * x ** np.arange(1, 201)
    print(f"  |x| = {x:.6f} ({tag:7s}): |term| at n = 200 is {t[-1]:.3e}")
print("\nverified: R = 1/e")

|a_n|^(1/n) at n = 10, 1e3, 2e4 : 2.5937424601, 2.7169239322, 2.7182138745
e                                = 2.7182818285
R = 1 / limsup                   = 0.3678886380   (1/e = 0.3678794412)
  |x| = 0.331091 (inside ): |term| at n = 200 is 4.286e-10
  |x| = 0.404667 (outside): |term| at n = 200 is 1.154e+08

verified: R = 1/e


### Problem L1.7 — Differentiation Preserves the Radius
**Source:** Stewart, *Calculus: Early Transcendentals*, Ch. 11

**Statement**

Let $f(x) = \sum_{n=0}^\infty a_n x^n$ have radius of convergence $R$. Prove that the term-by-term differentiated series $g(x) = \sum_{n=1}^\infty n a_n x^{n-1}$ has the exact same radius of convergence $R$.

**Intuition**
Differentiating brings down a factor of $n$. Since $\lim_{n\to\infty} \sqrt[n]{n} = 1$, polynomial growth factors $n^k$ do not alter the exponential growth rate of $a_n$, leaving $\limsup \lvert a_n \rvert^{\frac{1}{n}}$ invariant.

**Solution**
By the Cauchy-Hadamard Theorem, the radius of convergence $R$ of $\sum a_n x^n$ satisfies:

$$
\frac{1}{R} = \limsup_{n\to\infty} \lvert a_n \rvert^{\frac{1}{n}}
$$

For the differentiated series $\sum_{n=1}^\infty n a_n x^{n-1} = \sum_{k=0}^\infty (k+1) a_{k+1} x^k$, its radius of convergence $R'$ satisfies:

$$
\frac{1}{R'} = \limsup_{k\to\infty} \lvert (k+1) a_{k+1} \rvert^{\frac{1}{k}} = \limsup_{n\to\infty} \lvert n a_n \rvert^{\frac{1}{n-1}}
$$

Take the limit of $\lvert n a_n \rvert^{\frac{1}{n-1}}$:

$$
\lvert n a_n \rvert^{\frac{1}{n-1}} = \left( n^{\frac{1}{n-1}} \right) \cdot \left( \lvert a_n \rvert^{\frac{1}{n}} \right)^{\frac{n}{n-1}}
$$

Since $\lim_{n\to\infty} n^{\frac{1}{n-1}} = 1$ and $\lim_{n\to\infty} \frac{n}{n-1} = 1$:

$$
\frac{1}{R'} = 1 \cdot \limsup_{n\to\infty} \lvert a_n \rvert^{\frac{1}{n}} = \frac{1}{R}
$$

Therefore $R' = R$.

$$
\boxed{ R' = R }
$$

**Key takeaway**
Algebraic scale factors like $n^p$ do not affect the radius of convergence of a power series.

In [14]:
# L1.7 — differentiating term by term leaves limsup |a_n|^(1/n), hence R, unchanged.
from math import lgamma

ns = np.arange(1, 3001)
cases = {
    "a_n = 3^n":     ns * np.log(3.0),
    "a_n = 1 / n!":  -np.array([lgamma(int(n) + 1) for n in ns]),
    "a_n = n^5 2^n": 5 * np.log(ns) + ns * np.log(2.0),
}
print(f"{'coefficients':16s} {'limsup |a_n|^(1/n)':>20s} {'limsup |n a_n|^(1/(n-1))':>26s}")
for name, la in cases.items():
    r0 = np.exp(la[-1] / ns[-1])
    r1 = np.exp((np.log(ns[-1]) + la[-1]) / (ns[-1] - 1))
    print(f"{name:16s} {r0:20.10f} {r1:26.10f}")
    assert abs(r0 - r1) < 5e-3 * max(1.0, r0)
print("\nthe two limits agree in every case: R' = R")

coefficients       limsup |a_n|^(1/n)   limsup |n a_n|^(1/(n-1))
a_n = 3^n                3.0000000000               3.0091218559
a_n = 1 / n!             0.0009046085               0.0009049097
a_n = n^5 2^n            2.0268667474               2.0327638737

the two limits agree in every case: R' = R


### Problem L1.8 — A Fourth-Order Limit by Taylor Expansion
**Source:** Demidovich, *Problems in Analysis*, No. 1388

**Statement**

Evaluate the limit using Taylor expansions:

$$
L = \lim_{x\to 0} \frac{\cos x - e^{-\frac{x^2}{2}}}{x^4}
$$

**Intuition**
Expand both $\cos x$ and $e^{-\frac{x^2}{2}}$ up to $O(x^4)$ to isolate the leading non-zero term in the numerator.

**Solution**
Expand $\cos x$ near $x=0$:

$$
\cos x = 1 - \frac{x^2}{2!} + \frac{x^4}{4!} + O(x^6) = 1 - \frac{x^2}{2} + \frac{x^4}{24} + O(x^6)
$$

Expand $e^u$ for $u = -\frac{x^2}{2}$:

$$
e^{-\frac{x^2}{2}} = 1 + \left(-\frac{x^2}{2}\right) + \frac{1}{2!}\left(-\frac{x^2}{2}\right)^2 + O(x^6) = 1 - \frac{x^2}{2} + \frac{x^4}{8} + O(x^6)
$$

Subtract the two expansions:

$$
\begin{aligned} \cos x - e^{-\frac{x^2}{2}} &= \left( 1 - \frac{x^2}{2} + \frac{x^4}{24} \right) - \left( 1 - \frac{x^2}{2} + \frac{x^4}{8} \right) + O(x^6) \\ &= \left( \frac{1}{24} - \frac{1}{8} \right) x^4 + O(x^6) \\ &= \left( \frac{1 - 3}{24} \right) x^4 + O(x^6) = -\frac{1}{12} x^4 + O(x^6) \end{aligned}
$$

Substitute into the limit:

$$
L = \lim_{x\to 0} \frac{-\frac{1}{12}x^4 + O(x^6)}{x^4} = \lim_{x\to 0} \left( -\frac{1}{12} + O(x^2) \right) = -\frac{1}{12}
$$

$$
\boxed{ L = -\frac{1}{12} }
$$

**Key takeaway**
Taylor expansions eliminate repetitive application of L'Hôpital's Rule for higher-order limits.

In [15]:
# L1.8 — the limit (cos x - e^{-x^2/2}) / x^4 -> -1/12.
print("   x           quotient")
for x in (1e-1, 1e-2, 1e-3):
    q = (np.cos(x) - np.exp(-x ** 2 / 2)) / x ** 4
    print(f"{x:.0e}     {q:.12f}")
print(f"\n-1/12 = {-1/12:.12f}")
q = (np.cos(1e-2) - np.exp(-1e-4 / 2)) / 1e-8
assert abs(q + 1 / 12) < 1e-4
# the same answer symbolically
import sympy as sp
xs = sp.symbols('x')
print("sympy limit :", sp.limit((sp.cos(xs) - sp.exp(-xs ** 2 / 2)) / xs ** 4, xs, 0))
assert sp.limit((sp.cos(xs) - sp.exp(-xs ** 2 / 2)) / xs ** 4, xs, 0) == sp.Rational(-1, 12)

   x           quotient
1e-01     -0.083139146565
1e-02     -0.083331386236
1e-03     -0.083377749149

-1/12 = -0.083333333333


sympy limit : -1/12


### Problem L1.9 — Trigonometric Series Summation via Euler's Formula
**Source:** Apostol, *Calculus I*, Ch. 9

**Statement**

Find the closed-form sum of the infinite series for any $\theta \in \mathbb{R}$:

$$
S(\theta) = \sum_{n=0}^\infty \frac{\cos(n\theta)}{n!}
$$

**Intuition**
Recognize $\cos(n\theta)$ as the real part of $e^{in\theta} = (e^{i\theta})^n$. Summing $\frac{(e^{i\theta})^n}{n!}$ yields $e^{e^{i\theta}}$.

**Solution**
Consider the complex series:

$$
C(\theta) = \sum_{n=0}^\infty \frac{(e^{i\theta})^n}{n!}
$$

Since $\sum_{n=0}^\infty \frac{z^n}{n!} = e^z$ for all $z \in \mathbb{C}$, set $z = e^{i\theta} = \cos\theta + i\sin\theta$:

$$
C(\theta) = e^{\cos\theta + i\sin\theta} = e^{\cos\theta} \cdot e^{i\sin\theta}
$$

Apply Euler's formula to $e^{i\sin\theta}$:

$$
C(\theta) = e^{\cos\theta} \left[ \cos(\sin\theta) + i \sin(\sin\theta) \right]
$$

Taking the real part:

$$
S(\theta) = \operatorname{Re}(C(\theta)) = \sum_{n=0}^\infty \frac{\cos(n\theta)}{n!} = e^{\cos\theta} \cos(\sin\theta)
$$

$$
\boxed{ \sum_{n=0}^\infty \frac{\cos(n\theta)}{n!} = e^{\cos\theta} \cos(\sin\theta) }
$$

**Key takeaway**
Complex power series reduce oscillating trigonometric sum problems to exponential arithmetic.

In [16]:
# L1.9 — sum cos(n theta) / n! = e^{cos theta} cos(sin theta).
ks = np.arange(0, 60)
logfact = np.array([sum(np.log(j) for j in range(1, int(k) + 1)) for k in ks])
print("  theta      series sum          e^{cos t} cos(sin t)      |difference|")
for theta in (0.0, 0.7, 1.5, np.pi, 4.2):
    s = float(np.sum(np.cos(ks * theta) * np.exp(-logfact)))
    closed = np.exp(np.cos(theta)) * np.cos(np.sin(theta))
    print(f"{theta:7.3f}   {s:17.12f}   {closed:20.12f}   {abs(s-closed):.2e}")
    assert abs(s - closed) < 1e-12
print("\nverified for five angles, to machine precision")

  theta      series sum          e^{cos t} cos(sin t)      |difference|
  0.000      2.718281828459         2.718281828459   4.44e-16
  0.700      1.717999960952         1.717999960952   2.22e-16
  1.500      0.582166574705         0.582166574705   1.11e-16
  3.142      0.367879441171         0.367879441171   1.11e-16
  4.200      0.394196591663         0.394196591663   1.11e-16

verified for five angles, to machine precision


### Problem L1.10 — Cauchy Remainder for the Binomial Series
**Source:** Spivak, *Calculus*, Ch. 19

**Statement**

Show using the Cauchy remainder formula that the binomial series $(1+x)^{\frac{1}{2}} = \sum_{n=0}^\infty \binom{\frac{1}{2}}{n} x^n$ converges for all $x \in (-1, 0)$.

**Intuition**
The Lagrange remainder for $(1+x)^\alpha$ contains terms like $(1+c)^{\alpha - n - 1}$, which blow up near $x = -1$. The Cauchy remainder contains $\left(\frac{x-c}{1+c}\right)^n$, which remains bounded for $x \in (-1, 0)$.

**Solution**
The Cauchy remainder for $f(x) = (1+x)^\alpha$ centered at $a=0$ is:

$$
R_n(x) = \frac{f^{(n+1)}(c)}{n!} (x-c)^n x = \frac{\alpha(\alpha-1)\dots(\alpha-n)}{n!} (1+c)^{\alpha-n-1} (x-c)^n x
$$

For $\alpha = \frac{1}{2}$:

$$
R_n(x) = \frac{\frac{1}{2}\left(-\frac{1}{2}\right)\dots\left(\frac{1}{2}-n\right)}{n!} (1+c)^{-\frac{1}{2}} \left( \frac{x-c}{1+c} \right)^n x
$$

For $-1 \lt x \lt c \lt 0$:
1. $0 \lt \frac{c - x}{1 + c} \lt \frac{0 - x}{1 + x} = \frac{\lvert x \rvert}{1 - \lvert x \rvert}$. But more directly, since $-1 \lt x \lt c \lt 0$, $0 \lt \frac{c-x}{1+c} = \frac{c+1 - (1+x)}{1+c} = 1 - \frac{1+x}{1+c} \lt 1 - (1+x) = \lvert x \rvert$.
   Thus $\left\vert\frac{x-c}{1+c}\right\vert \lt \lvert x \rvert \lt 1$.
2. $(1+c)^{-\frac{1}{2}} \lt (1+x)^{-\frac{1}{2}}$, which is a fixed constant independent of $n$.

Therefore, with $K = (1+x)^{-\frac{1}{2}}$ a constant independent of $n$, and writing
$c_n = \dfrac{\lvert \alpha(\alpha-1)\cdots(\alpha-n) \rvert}{n!}$ for the coefficient factor,

$$
\lvert R_n(x) \rvert \le K \, c_n \, \lvert x \rvert^{n+1} .
$$

Now $c_n \lvert x \rvert^{n+1} \to 0$ by the ratio test: since

$$
\frac{c_{n+1}}{c_n} = \frac{\lvert \alpha - n - 1 \rvert}{n+1} \longrightarrow 1 ,
$$

the ratio of consecutive terms of $c_n \lvert x \rvert^{n+1}$ tends to $\lvert x \rvert \lt 1$, so
those terms decay geometrically. Hence $R_n(x) \to 0$ for every $x \in (-1, 0)$, and the binomial
series converges to $(1+x)^{\frac{1}{2}}$ on the whole of that interval.

$$
\boxed{ \text{Cauchy remainder bounds } \left\vert\frac{x-c}{1+c}\right\vert \lt \lvert x \rvert \lt 1 \text{ for } x \in (-1, 0), \text{ proving convergence.} }
$$

**Key takeaway**
The Cauchy remainder form is superior to Lagrange when proving convergence of binomial series near negative boundaries.

In [17]:
# L1.10 — the Cauchy remainder converges on (-1, 0) where the Lagrange bound diverges.
def binom_coeff(alpha, n):
    out = 1.0
    for j in range(n):
        out *= (alpha - j) / (j + 1)
    return out

alpha, x = 0.5, -0.9
true = (1 + x) ** alpha
print(f"{'n':>4s} {'|R_n| true':>14s} {'Cauchy bound':>16s} {'Lagrange bound':>18s}")
for n in (2, 4, 8, 16, 32, 48):
    Pn = sum(binom_coeff(alpha, k) * x ** k for k in range(n + 1))
    Rn = abs(true - Pn)
    cau = (n + 1) * abs(binom_coeff(alpha, n + 1)) * max(1.0, (1 + x) ** (alpha - 1)) * abs(x) ** (n + 1)
    lag = abs(binom_coeff(alpha, n + 1)) * (1 + x) ** (alpha - n - 1) * abs(x) ** (n + 1)
    print(f"{n:4d} {Rn:14.3e} {cau:16.3e} {lag:18.3e}")
    assert Rn <= cau
n = 48
Pn = sum(binom_coeff(alpha, k) * x ** k for k in range(n + 1))
cau = (n + 1) * abs(binom_coeff(alpha, n + 1)) * max(1.0, (1 + x) ** (alpha - 1)) * abs(x) ** (n + 1)
lag = abs(binom_coeff(alpha, n + 1)) * (1 + x) ** (alpha - n - 1) * abs(x) ** (n + 1)
assert abs(true - Pn) < cau < lag
print("\nthe Cauchy bound tends to 0 while the Lagrange bound diverges: that is why Theorem 4.3 exists")

   n     |R_n| true     Cauchy bound     Lagrange bound
   2      1.325e-01        4.322e-01          1.441e+01
   4      6.133e-02        2.553e-01          5.106e+02
   8      2.094e-02        1.203e-01          1.337e+06
  16      4.286e-03        3.690e-02          2.171e+13
  32      3.471e-04        4.854e-03          1.471e+28
  48      3.829e-05        7.354e-04          1.501e+43

the Cauchy bound tends to 0 while the Lagrange bound diverges: that is why Theorem 4.3 exists


### Problem L1.11 — Gregory's Series for $\pi$
**Source:** Spivak, *Calculus*, Ch. 19

**Statement**

(a) Derive the Maclaurin series for $\arctan(x)$ by integrating the series for $\frac{1}{1+x^2}$.  
(b) Use Abel's Theorem to prove Gregory's formula: $\sum_{n=0}^\infty \frac{(-1)^n}{2n+1} = \frac{\pi}{4}$.

**Intuition**
Since $\frac{d}{dx}\arctan(x) = \frac{1}{1+x^2} = \sum (-1)^n x^{2n}$ for $\lvert x \rvert \lt 1$, integrating term-by-term yields the series for $\arctan(x)$. Abel's theorem extends equality to the boundary $x=1$.

**Solution**
(a) For $\lvert x \rvert \lt 1$:

$$
\frac{1}{1+x^2} = \sum_{n=0}^\infty (-1)^n x^{2n}
$$

Integrate term-by-term from $0$ to $x$:

$$
\arctan(x) = \int_0^x \sum_{n=0}^\infty (-1)^n t^{2n} \, dt = \sum_{n=0}^\infty (-1)^n \int_0^x t^{2n} \, dt = \sum_{n=0}^\infty \frac{(-1)^n x^{2n+1}}{2n+1}
$$

Radius of convergence $R = 1$.

(b) At $x = 1$, the numerical series is $\sum_{n=0}^\infty \frac{(-1)^n}{2n+1}$.  
This is an alternating series with $b_n = \frac{1}{2n+1} \to 0$ monotonically. By the Alternating Series Test, it converges.

By **Abel's Theorem**, $f(x) = \sum_{n=0}^\infty \frac{(-1)^n x^{2n+1}}{2n+1}$ is continuous on $[0, 1]$.  
Thus:

$$
\sum_{n=0}^\infty \frac{(-1)^n}{2n+1} = \lim_{x \to 1^-} \arctan(x) = \arctan(1) = \frac{\pi}{4}
$$

$$
\boxed{ \sum_{n=0}^\infty \frac{(-1)^n}{2n+1} = \frac{\pi}{4} }
$$

**Key takeaway**
Abel's theorem rigorously justifies taking the boundary limit of a conditionally convergent power series.

In [18]:
# L1.11 — Gregory's series converges to pi/4 (slowly), matching arctan on the interior.
for N in (10, 100, 1000, 10000):
    k = np.arange(N)
    partial = float(np.sum((-1.0) ** k / (2 * k + 1)))
    print(f"N = {N:6d}: partial = {partial:.10f}   error = {abs(partial - np.pi/4):.3e}"
          f"   1/(4N) = {1/(4*N):.3e}")
print(f"\npi/4 = {np.pi/4:.12f}")
for x in (0.9, 0.99, 0.999):
    k = np.arange(20000)
    s = float(np.sum((-1.0) ** k * x ** (2 * k + 1) / (2 * k + 1)))
    print(f"  interior x = {x:.3f}: series = {s:.12f}   arctan(x) = {np.arctan(x):.12f}")
    assert abs(s - np.arctan(x)) < 1e-10
k = np.arange(200000)
assert abs(float(np.sum((-1.0) ** k / (2 * k + 1))) - np.pi / 4) < 1e-5
print("\nverified: the boundary value is pi/4, reached as the Abel limit of arctan")

N =     10: partial = 0.7604599047   error = 2.494e-02   1/(4N) = 2.500e-02
N =    100: partial = 0.7828982259   error = 2.500e-03   1/(4N) = 2.500e-03
N =   1000: partial = 0.7851481635   error = 2.500e-04   1/(4N) = 2.500e-04
N =  10000: partial = 0.7853731634   error = 2.500e-05   1/(4N) = 2.500e-05

pi/4 = 0.785398163397
  interior x = 0.900: series = 0.732815101787   arctan(x) = 0.732815101787
  interior x = 0.990: series = 0.780373080067   arctan(x) = 0.780373080067
  interior x = 0.999: series = 0.784897913314   arctan(x) = 0.784897913314

verified: the boundary value is pi/4, reached as the Abel limit of arctan


### Problem L1.12 — Alternating Harmonic Series via Abel's Theorem
**Source:** Apostol, *Calculus I*, Ch. 9

**Statement**

Starting from the power series $\ln(1+x) = \sum_{n=1}^\infty \frac{(-1)^{n-1}}{n} x^n$ for $\lvert x \rvert \lt 1$, prove rigorously that:

$$
\sum_{n=1}^\infty \frac{(-1)^{n-1}}{n} = 1 - \frac{1}{2} + \frac{1}{3} - \frac{1}{4} + \dots = \ln 2
$$

**Intuition**
The series converges conditionally at $x = 1$. Abel's theorem guarantees that the limit of $\ln(1+x)$ as $x \to 1^-$ equals the sum of the series at $x=1$.

**Solution**
1. **Interior Equality**:  
   For $\lvert x \rvert \lt 1$, $\ln(1+x) = \sum_{n=1}^\infty \frac{(-1)^{n-1} x^n}{n}$.

2. **Boundary Convergence at $x=1$**:  
   At $x=1$, the series is $\sum_{n=1}^\infty \frac{(-1)^{n-1}}{n}$.  
   The terms $a_n = \frac{1}{n}$ decrease monotonically to $0$ ($a_{n+1} \lt a_n$ and $\lim a_n = 0$).  
   By the Leibniz Alternating Series Test, $\sum_{n=1}^\infty \frac{(-1)^{n-1}}{n}$ converges to some real value $S$.

3. **Application of Abel's Theorem**:  
   Abel's Theorem states that if a power series converges at $x=R$, the sum function $f(x)$ is continuous on $[0, R]$.  
   Here $R = 1$:

$$
   S = \sum_{n=1}^\infty \frac{(-1)^{n-1}}{n} = \lim_{x \to 1^-} \ln(1+x) = \ln(1+1) = \ln 2
$$

$$
\boxed{ \sum_{n=1}^\infty \frac{(-1)^{n-1}}{n} = \ln 2 }
$$

**Key takeaway**
Abel's theorem transfers function limits to series limits at boundary points.

In [19]:
# L1.12 — the alternating harmonic series sums to ln 2, as Abel's theorem predicts.
for N in (10, 1000, 100000, 2000000):
    n = np.arange(1, N + 1)
    partial = float(np.sum((-1.0) ** (n - 1) / n))
    print(f"N = {N:8d}: partial = {partial:.10f}   error = {abs(partial - np.log(2)):.3e}"
          f"   1/(2N) = {1/(2*N):.3e}")
print(f"\nln 2 = {np.log(2):.12f}")
for x in (0.9, 0.99, 0.999):
    n = np.arange(1, 50001)
    s = float(np.sum((-1.0) ** (n - 1) * x ** n / n))
    print(f"  interior x = {x:.3f}: series = {s:.12f}   ln(1+x) = {np.log1p(x):.12f}")
    assert abs(s - np.log1p(x)) < 1e-9
n = np.arange(1, 2000001)
assert abs(float(np.sum((-1.0) ** (n - 1) / n)) - np.log(2)) < 1e-6

N =       10: partial = 0.6456349206   error = 4.751e-02   1/(2N) = 5.000e-02
N =     1000: partial = 0.6926474306   error = 4.998e-04   1/(2N) = 5.000e-04
N =   100000: partial = 0.6931421806   error = 5.000e-06   1/(2N) = 5.000e-06


N =  2000000: partial = 0.6931469306   error = 2.500e-07   1/(2N) = 2.500e-07

ln 2 = 0.693147180560


  interior x = 0.900: series = 0.641853886172   ln(1+x) = 0.641853886172
  interior x = 0.990: series = 0.688134638736   ln(1+x) = 0.688134638736
  interior x = 0.999: series = 0.692647055518   ln(1+x) = 0.692647055518


## L2 — Applications (AI/ML and Physics)

Twelve applications. Five are physics — L2.1, L2.2, L2.6, L2.8 and L2.10 — six are machine learning, and L2.12 is the floating-point step-size problem.

### Problem L2.1 — Physics — Relativistic Kinetic Energy Correction
**Source:** Classical Mechanics / Relativistic Physics

**Statement**

The total energy of a relativistic particle is $E = \gamma m c^2 = \frac{m c^2}{\sqrt{1 - \frac{v^2}{c^2}}}$.  
(a) Derive the 2nd non-zero Taylor correction term in $\beta = \frac{v}{c}$.  
(b) Estimate the relative error of using Newtonian kinetic energy $E_k \approx \frac{1}{2}mv^2$ at $v = 0.1c$.

**Intuition**
Expand $(1 - \beta^2)^{-\frac{1}{2}}$ using the binomial series. Subtracting the rest mass $mc^2$ yields the kinetic energy $E_k$.

**Solution**
(a) Let $x = \beta^2 = \left(\frac{v}{c}\right)^2$. Binomial expansion of $(1-x)^{-\frac{1}{2}}$:

$$
(1-x)^{-\frac{1}{2}} = 1 + \frac{1}{2}x + \frac{3}{8}x^2 + O(x^3)
$$

Thus:

$$
E = m c^2 \left( 1 + \frac{1}{2}\frac{v^2}{c^2} + \frac{3}{8}\frac{v^4}{c^4} + O\left(\frac{v^6}{c^6}\right) \right)
$$

Kinetic energy $E_k = E - m c^2$:

$$
E_k = \frac{1}{2} m v^2 + \frac{3}{8} m \frac{v^4}{c^2} + O\left(\frac{v^6}{c^4}\right)
$$

(b) Newtonian approximation $E_{Newton} = \frac{1}{2} m v^2$.  
The leading error term is $\Delta E_k \approx \frac{3}{8} m \frac{v^4}{c^2}$.  
Relative error:

$$
\frac{\Delta E_k}{E_{Newton}} \approx \frac{\frac{3}{8} m v^4 / c^2}{\frac{1}{2} m v^2} = \frac{3}{4} \left( \frac{v}{c} \right)^2
$$

For $v = 0.1c$:

$$
\text{Relative Error} \approx \frac{3}{4} (0.1)^2 = 0.75 \times 0.01 = 0.0075 = 0.75\%
$$

$$
\boxed{ E_k = \frac{1}{2}mv^2 + \frac{3}{8}m\frac{v^4}{c^2} + O\left(\frac{v^6}{c^4}\right), \quad \text{Relative Error at } 0.1c \approx 0.75\% }
$$

**Key takeaway**
Classical mechanics is precisely the 1st-order Taylor approximation of special relativity in the limit $\frac{v}{c} \to 0$.

In [20]:
# L2.1 — the quartic correction to the kinetic energy, and the 0.75% relative error at v = 0.1c.
m, c = 1.0, 1.0
print(f"{'beta':>6s} {'exact E_k':>16s} {'Newton':>16s} {'Newton + 3/8 term':>20s}"
      f" {'rel. error of Newton':>22s} {'3/4 beta^2':>12s}")
for beta in (0.01, 0.05, 0.1, 0.2):
    v = beta * c
    exact = m * c ** 2 * (1 / np.sqrt(1 - beta ** 2) - 1)
    newton = 0.5 * m * v ** 2
    two_term = newton + 3 / 8 * m * v ** 4 / c ** 2
    rel = (exact - newton) / newton
    print(f"{beta:6.2f} {exact:16.10f} {newton:16.10f} {two_term:20.10f}"
          f" {rel:22.6f} {0.75*beta**2:12.6f}")
rel01 = (1 / np.sqrt(1 - 0.01) - 1 - 0.005) / 0.005
print(f"\nrelative error at v = 0.1c : {rel01:.6f} = {100*rel01:.4f}%   (predicted 0.75%)")
print("the 0.006 percentage-point gap is the next term, O(beta^4); it shrinks as beta^2:")
for beta in (0.1, 0.05, 0.025):
    rel = (1 / np.sqrt(1 - beta ** 2) - 1 - beta ** 2 / 2) / (beta ** 2 / 2)
    print(f"  beta = {beta:6.3f}: (rel - 0.75 beta^2) / beta^4 = "
          f"{(rel - 0.75*beta**2)/beta**4:.6f}   (bounded -> the gap is O(beta^4))")
    assert abs((rel - 0.75 * beta ** 2) / beta ** 4 - 0.625) < 0.06
assert abs(rel01 - 0.0075) < 1e-4
print("verified: the leading Newtonian relative error is (3/4) beta^2 = 0.75% at beta = 0.1")

  beta        exact E_k           Newton    Newton + 3/8 term   rel. error of Newton   3/4 beta^2
  0.01     0.0000500038     0.0000500000         0.0000500038               0.000075     0.000075
  0.05     0.0012523486     0.0012500000         0.0012523438               0.001879     0.001875
  0.10     0.0050378153     0.0050000000         0.0050375000               0.007563     0.007500
  0.20     0.0206207262     0.0200000000         0.0206000000               0.031036     0.030000

relative error at v = 0.1c : 0.007563 = 0.7563%   (predicted 0.75%)
the 0.006 percentage-point gap is the next term, O(beta^4); it shrinks as beta^2:
  beta =  0.100: (rel - 0.75 beta^2) / beta^4 = 0.630518   (bounded -> the gap is O(beta^4))
  beta =  0.050: (rel - 0.75 beta^2) / beta^4 = 0.626370   (bounded -> the gap is O(beta^4))
  beta =  0.025: (rel - 0.75 beta^2) / beta^4 = 0.625341   (bounded -> the gap is O(beta^4))
verified: the leading Newtonian relative error is (3/4) beta^2 = 0.75% at beta =

### Problem L2.2 — Physics — Anharmonic Pendulum Period
**Source:** Marion & Thornton, *Classical Dynamics of Particles and Systems*

**Statement**

The potential energy of a simple pendulum is $V(\theta) = m g l (1 - \cos\theta)$.  
(a) Taylor expand $V(\theta)$ up to $O(\theta^4)$.  
(b) Derive the leading correction factor to the pendulum period $\frac{T}{T_0} \approx 1 + \frac{1}{16}\theta_0^2$ for amplitude $\theta_0$.

**Intuition**
Small angle approximation assumes $\cos\theta \approx 1 - \frac{\theta^2}{2}$, giving a simple harmonic oscillator. Including $\frac{\theta^4}{24}$ introduces non-linear restoring forces, increasing the period at larger amplitudes.

**Solution**
(a) Maclaurin expansion of $\cos\theta$:

$$
\cos\theta = 1 - \frac{\theta^2}{2} + \frac{\theta^4}{24} + O(\theta^6)
$$

Substituting into $V(\theta)$:

$$
V(\theta) = m g l \left( \frac{\theta^2}{2} - \frac{\theta^4}{24} \right) + O(\theta^6)
$$

(b) Conservation of energy gives $\frac{1}{2} m l^2 \dot{\theta}^2 + V(\theta) = V(\theta_0)$:

$$
\dot{\theta} = \sqrt{\frac{2 g}{l} (\cos\theta - \cos\theta_0)}
$$

The exact period is:

$$
T = 4 \sqrt{\frac{l}{2g}} \int_0^{\theta_0} \frac{d\theta}{\sqrt{\cos\theta - \cos\theta_0}}
$$

Use the half-angle identity $\cos\theta - \cos\theta_0 = 2\left(\sin^2\frac{\theta_0}{2} - \sin^2\frac{\theta}{2}\right)$ and substitute
$\sin\frac{\theta}{2} = \sin\frac{\theta_0}{2}\sin u$, so $\theta=0 \Rightarrow u=0$ and $\theta=\theta_0 \Rightarrow u = \pi/2$, and
$\frac{1}{2}\cos\frac{\theta}{2}\,d\theta = \sin\frac{\theta_0}{2}\cos u\,du$. This turns the integral into the complete elliptic
integral of the first kind, exactly (no approximation yet):

$$
T = 4\sqrt{\frac{l}{g}} \int_0^{\pi/2} \frac{du}{\sqrt{1 - k^2\sin^2 u}} = 4\sqrt{\frac{l}{g}}\, K(k), \qquad k = \sin\frac{\theta_0}{2}
$$

Now Taylor expand $K(k)$ in $k$ using $(1-x)^{-1/2} \approx 1 + \frac{x}{2} + \frac{3}{8}x^2$ inside the integrand and
$\int_0^{\pi/2}\sin^2 u\,du = \pi/4$:

$$
K(k) = \frac{\pi}{2}\left(1 + \frac{1}{4}k^2 + O(k^4)\right)
$$

With $k = \sin(\theta_0/2) \approx \theta_0/2$ for small $\theta_0$, so $k^2 \approx \theta_0^2/4$:

$$
T \approx 4\sqrt{\frac{l}{g}} \cdot \frac{\pi}{2}\left(1 + \frac{\theta_0^2}{16}\right) = T_0\left(1 + \frac{1}{16}\theta_0^2\right), \quad \text{where } T_0 = 2\pi\sqrt{\frac{l}{g}}
$$

$$
\boxed{ V(\theta) \approx mgl\left(\frac{\theta^2}{2} - \frac{\theta^4}{24}\right), \quad \frac{T}{T_0} \approx 1 + \frac{1}{16}\theta_0^2 }
$$

**Key takeaway**
Higher-order Taylor terms explain physical non-linearities like amplitude-dependent oscillation periods.

In [21]:
# L2.2 — the pendulum period correction T/T_0 = 1 + theta_0^2/16, against the exact elliptic integral.
from scipy.special import ellipk

print(f"{'theta_0 (rad)':>14s} {'exact T/T_0':>14s} {'1 + th^2/16':>14s} {'difference':>14s}")
for th0 in (0.05, 0.1, 0.2, 0.4, 0.8):
    kk = np.sin(th0 / 2)
    exact = 2 / np.pi * ellipk(kk ** 2)      # scipy's ellipk takes the parameter m = k^2
    approx = 1 + th0 ** 2 / 16
    print(f"{th0:14.2f} {exact:14.10f} {approx:14.10f} {exact-approx:14.3e}")
    assert abs(exact - approx) < 0.02 * th0 ** 2
# the quartic term of the potential
th = np.linspace(-0.6, 0.6, 401)
V_exact = 1 - np.cos(th)
V_quart = th ** 2 / 2 - th ** 4 / 24
print(f"\nmax |V/(mgl) - (th^2/2 - th^4/24)| on |th| <= 0.6 : {np.max(np.abs(V_exact - V_quart)):.3e}")
assert np.max(np.abs(V_exact - V_quart)) < 1e-4
print("verified: the elliptic-integral period matches 1 + theta_0^2/16 to O(theta_0^4)")

 theta_0 (rad)    exact T/T_0    1 + th^2/16     difference
          0.05   1.0001562724   1.0001562500      2.238e-08
          0.10   1.0006253583   1.0006250000      3.583e-07
          0.20   1.0025057442   1.0025000000      5.744e-06
          0.40   1.0100926393   1.0100000000      9.264e-05
          0.80   1.0415312470   1.0400000000      1.531e-03

max |V/(mgl) - (th^2/2 - th^4/24)| on |th| <= 0.6 : 6.439e-05
verified: the elliptic-integral period matches 1 + theta_0^2/16 to O(theta_0^4)


### Problem L2.3 — AI/ML — GELU Activation Expansion
**Source:** Hendrycks & Gimpel (2016), *Gaussian Error Linear Units (GELUs)*

**Statement**

The GELU activation function is defined as $f(x) = x \Phi(x) = x \int_{-\infty}^x \frac{1}{\sqrt{2\pi}} e^{-\frac{t^2}{2}} dt$.  
Derive the Maclaurin expansion of $f(x)$ through the $x^4$ term.

**Intuition**
Split $\Phi(x) = \frac{1}{2} + \frac{1}{\sqrt{2\pi}} \int_0^x e^{-\frac{t^2}{2}} dt$. Taylor expand $e^{-\frac{t^2}{2}}$ inside the integral and multiply by $x$.

**Solution**
1. **Expand Gaussian Integral**:

$$
   e^{-\frac{t^2}{2}} = 1 - \frac{t^2}{2} + \frac{t^4}{8} - O(t^6)
$$

   Integrate term-by-term:

$$
   \int_0^x e^{-\frac{t^2}{2}} dt = x - \frac{x^3}{6} + \frac{x^5}{40} - O(x^7)
$$

2. **Construct $\Phi(x)$**:

$$
   \Phi(x) = \frac{1}{2} + \frac{1}{\sqrt{2\pi}} \left( x - \frac{x^3}{6} + O(x^5) \right)
$$

3. **Construct GELU $f(x) = x \Phi(x)$**:

$$
   \begin{aligned} f(x) &= x \left[ \frac{1}{2} + \frac{1}{\sqrt{2\pi}} \left( x - \frac{x^3}{6} + O(x^5) \right) \right] \\ &= \frac{1}{2}x + \frac{1}{\sqrt{2\pi}} x^2 - \frac{1}{6\sqrt{2\pi}} x^4 + O(x^6) \end{aligned}
$$

$$
\boxed{ f(x) = \frac{1}{2}x + \frac{1}{\sqrt{2\pi}} x^2 - \frac{1}{6\sqrt{2\pi}} x^4 + O(x^6) }
$$

**Key takeaway**
Near $x=0$, GELU behaves as a smooth mixture of a linear term $\frac{1}{2}x$ and a quadratic convex term $\frac{1}{\sqrt{2\pi}}x^2$.

In [22]:
# L2.3 — the GELU expansion x/2 + x^2/sqrt(2 pi) - x^4/(6 sqrt(2 pi)).
from scipy.stats import norm

s2pi = np.sqrt(2 * np.pi)
gelu = lambda x: x * norm.cdf(x)
approx = lambda x: x / 2 + x ** 2 / s2pi - x ** 4 / (6 * s2pi)
print(f"{'x':>7s} {'GELU exact':>16s} {'quartic model':>16s} {'|error|':>12s} {'|error|/x^6':>14s}")
for x in (0.4, 0.2, 0.1, 0.05):
    e = abs(gelu(x) - approx(x))
    print(f"{x:7.2f} {gelu(x):16.12f} {approx(x):16.12f} {e:12.3e} {e/x**6:14.4f}")
    assert e < 0.02 * abs(x) ** 6 + 1e-15
print(f"\ncoefficients: 1/2 = {0.5}, 1/sqrt(2pi) = {1/s2pi:.10f}, -1/(6 sqrt(2pi)) = {-1/(6*s2pi):.10f}")
print("the error / x^6 column is bounded, confirming the omitted term is O(x^6)")

      x       GELU exact    quartic model      |error|    |error|/x^6
   0.40   0.262168696644   0.262128611135    4.009e-05         0.0098
   0.20   0.115851941888   0.115851306608    6.353e-07         0.0099
   0.10   0.053982783728   0.053982773766    9.962e-09         0.0100
   0.05   0.025996940292   0.025996940136    1.558e-10         0.0100

coefficients: 1/2 = 0.5, 1/sqrt(2pi) = 0.3989422804, -1/(6 sqrt(2pi)) = -0.0664903801
the error / x^6 column is bounded, confirming the omitted term is O(x^6)


### Problem L2.4 — AI/ML — Second-Order Log-Sum-Exp Model
**Source:** Goodfellow et al., *Deep Learning*, Ch. 4

**Statement**

Let $f(x_1, x_2) = \operatorname{LSE}(x_1, x_2) = \ln(e^{x_1} + e^{x_2})$.  
Compute the 2nd-order multivariable Taylor polynomial of $f$ centered at $(0, 0)$.

**Intuition**
LSE is the smooth approximation to $\max(x_1, x_2)$. At $(0,0)$, $e^0 + e^0 = 2$, so $f(0,0) = \ln 2$. We compute gradient and Hessian at origin.

**Solution**
1. **Value at origin**: $f(0,0) = \ln(1+1) = \ln 2$.

2. **First Derivatives**:

$$
   \frac{\partial f}{\partial x_1} = \frac{e^{x_1}}{e^{x_1} + e^{x_2}} \implies \frac{\partial f}{\partial x_1}(0,0) = \frac{1}{2}
$$

$$
   \frac{\partial f}{\partial x_2} = \frac{e^{x_2}}{e^{x_1} + e^{x_2}} \implies \frac{\partial f}{\partial x_2}(0,0) = \frac{1}{2}
$$

3. **Second Derivatives**:

$$
   \frac{\partial^2 f}{\partial x_1^2} = \frac{e^{x_1}(e^{x_1}+e^{x_2}) - e^{2x_1}}{(e^{x_1}+e^{x_2})^2} = \frac{e^{x_1+x_2}}{(e^{x_1}+e^{x_2})^2} \implies \frac{\partial^2 f}{\partial x_1^2}(0,0) = \frac{1}{4}
$$

$$
   \frac{\partial^2 f}{\partial x_2^2}(0,0) = \frac{1}{4}
$$

$$
   \frac{\partial^2 f}{\partial x_1 \partial x_2} = \frac{-e^{x_1+x_2}}{(e^{x_1}+e^{x_2})^2} \implies \frac{\partial^2 f}{\partial x_1 \partial x_2}(0,0) = -\frac{1}{4}
$$

4. **Construct 2nd-order Taylor Polynomial**:

$$
   P_2(x_1, x_2) = \ln 2 + \frac{1}{2}(x_1 + x_2) + \frac{1}{2} \left[ \frac{1}{4}x_1^2 - 2\frac{1}{4}x_1 x_2 + \frac{1}{4}x_2^2 \right]
$$

$$
   P_2(x_1, x_2) = \ln 2 + \frac{1}{2}(x_1 + x_2) + \frac{1}{8}(x_1 - x_2)^2
$$

$$
\boxed{ P_2(x_1, x_2) = \ln 2 + \frac{1}{2}(x_1 + x_2) + \frac{1}{8}(x_1 - x_2)^2 }
$$

**Key takeaway**
The curvature of Log-Sum-Exp penalizes differences $(x_1 - x_2)^2$, acting as a smooth local parabolic bowl around equality.

In [23]:
# L2.4 — the second-order model of log-sum-exp at the origin.
lse = lambda a, b: np.log(np.exp(a) + np.exp(b))
T2 = lambda a, b: np.log(2) + 0.5 * (a + b) + (a - b) ** 2 / 8
print(f"{'(x1, x2)':>16s} {'LSE exact':>16s} {'T_2':>16s} {'|error|':>12s}")
for a, b in [(0.2, -0.1), (0.05, 0.05), (-0.1, 0.3), (0.02, 0.01)]:
    e = abs(lse(a, b) - T2(a, b))
    r = np.hypot(a, b)
    print(f"({a:6.2f},{b:6.2f}) {lse(a,b):16.12f} {T2(a,b):16.12f} {e:12.3e}")
    assert e < 0.2 * r ** 3 + 1e-14
# numerical gradient and Hessian at the origin
h = 1e-5
g = [(lse(h, 0) - lse(-h, 0)) / (2 * h), (lse(0, h) - lse(0, -h)) / (2 * h)]
H11 = (lse(h, 0) - 2 * lse(0, 0) + lse(-h, 0)) / h ** 2
H12 = (lse(h, h) - lse(h, -h) - lse(-h, h) + lse(-h, -h)) / (4 * h ** 2)
print(f"\ngradient at 0 : {g}   (exact [0.5, 0.5])")
print(f"Hessian H11 = {H11:.6f}, H12 = {H12:.6f}   (exact 0.25 and -0.25)")
assert abs(g[0] - 0.5) < 1e-6 and abs(H11 - 0.25) < 1e-4 and abs(H12 + 0.25) < 1e-4

        (x1, x2)        LSE exact              T_2      |error|
(  0.20, -0.10)   0.754355244469   0.754397180560    4.194e-05
(  0.05,  0.05)   0.743147180560   0.743147180560    0.000e+00
( -0.10,  0.30)   0.813015252400   0.813147180560    1.319e-04
(  0.02,  0.01)   0.708159680508   0.708159680560    5.208e-11

gradient at 0 : [np.float64(0.4999999999921733), np.float64(0.4999999999921733)]   (exact [0.5, 0.5])
Hessian H11 = 0.250000, H12 = -0.250000   (exact 0.25 and -0.25)


### Problem L2.5 — AI/ML — The Newton-Raphson Step
**Source:** Nocedal & Wright, *Numerical Optimization*

**Statement**

Let $L(w)$ be a twice continuously differentiable loss function on $\mathbb{R}^d$. Derive the optimal update step $\Delta w$ from the second-order Taylor approximation of $L(w_0 + \Delta w)$.

**Intuition**
Approximate $L(w_0 + \Delta w)$ locally by a quadratic model. Finding the minimum of this quadratic model by setting its gradient with respect to $\Delta w$ to zero yields Newton's step.

**Solution**
1. **Quadratic Taylor Model**:

$$
   m(\Delta w) = L(w_0) + \nabla L(w_0)^T \Delta w + \frac{1}{2} \Delta w^T H(w_0) \Delta w
$$

   where $H(w_0) = \nabla^2 L(w_0)$ is the $d \times d$ symmetric Hessian matrix.

2. **Optimization Condition**:  
   To find the step $\Delta w$ that minimizes $m(\Delta w)$, compute the gradient with respect to $\Delta w$:

$$
   \nabla_{\Delta w} m(\Delta w) = \nabla L(w_0) + H(w_0) \Delta w
$$

3. **Solve for $\Delta w$**:  
   Set the gradient to zero:

$$
   \nabla L(w_0) + H(w_0) \Delta w = 0 \implies H(w_0) \Delta w = -\nabla L(w_0)
$$

   Assuming $H(w_0)$ is positive-definite (invertible):

$$
   \Delta w^\ast = - H(w_0)^{-1} \nabla L(w_0)
$$

$$
\boxed{ \Delta w^\ast = - H(w_0)^{-1} \nabla L(w_0) }
$$

**Key takeaway**
Newton's method uses 2nd-order Taylor expansion to leap directly to the minimum of the local quadratic approximation.

In [24]:
# L2.5 — the Newton step solves the quadratic model exactly, on a small quadratic loss.
H = np.array([[4.0, 1.0], [1.0, 3.0]])
b = np.array([1.0, -2.0])
L = lambda w: 0.5 * w @ H @ w + b @ w
grad = lambda w: H @ w + b

w0 = np.array([0.7, -0.4])
step = -np.linalg.solve(H, grad(w0))
w1 = w0 + step
print("H =\n", H)
print("w0        =", w0, "  L(w0) =", f"{L(w0):.10f}")
print("Newton step =", step)
print("w1        =", w1, "  L(w1) =", f"{L(w1):.10f}")
print("gradient at w1 =", grad(w1), " (zero: one step is exact for a quadratic)")
assert np.allclose(grad(w1), 0.0, atol=1e-12)
# the model value matches the true loss because the loss IS the quadratic model here
model = L(w0) + grad(w0) @ step + 0.5 * step @ H @ step
print(f"\nquadratic model at the step = {model:.12f}   true L(w1) = {L(w1):.12f}")
assert abs(model - L(w1)) < 1e-12
print("verified: Delta w* = -H^{-1} grad L minimises the second-order model")

H =
 [[4. 1.]
 [1. 3.]]
w0        = [ 0.7 -0.4]   L(w0) = 2.4400000000
Newton step = [-1.1545  1.2182]
w1        = [-0.4545  0.8182]   L(w1) = -1.0454545455
gradient at w1 = [0. 0.]  (zero: one step is exact for a quadratic)

quadratic model at the step = -1.045454545455   true L(w1) = -1.045454545455
verified: Delta w* = -H^{-1} grad L minimises the second-order model


### Problem L2.6 — Physics — van der Waals Virial Coefficients
**Source:** Reif, *Fundamentals of Statistical and Thermal Physics*

**Statement**

The van der Waals equation of state is $P = \frac{RT}{V-b} - \frac{a}{V^2}$.  
Express $P$ as a power series in terms of molar density $\rho = \frac{1}{V}$ up to $O(\rho^3)$ to find the second and third virial coefficients $B_2(T)$ and $B_3(T)$.

**Intuition**
Ideal gas law is $P = \rho RT$. Real gas interactions introduce non-linear density terms. Taylor expand $\frac{1}{V-b} = \frac{\rho}{1-b\rho}$ for small $\rho$.

**Solution**
Substitute $\rho = \frac{1}{V}$:

$$
P = \frac{RT \rho}{1 - b\rho} - a \rho^2
$$

Expand $(1 - b\rho)^{-1}$ in powers of $\rho$ for $b\rho \lt 1$:

$$
\frac{1}{1 - b\rho} = 1 + b\rho + b^2 \rho^2 + O(\rho^3)
$$

Substitute into $P$:

$$
\begin{aligned} P &= RT \rho \left( 1 + b\rho + b^2 \rho^2 + O(\rho^3) \right) - a \rho^2 \\ &= RT \rho + (RT b - a)\rho^2 + RT b^2 \rho^3 + O(\rho^4) \end{aligned}
$$

Standard Virial Expansion format $P = RT \left( \rho + B_2(T)\rho^2 + B_3(T)\rho^3 + \dots \right)$:

$$
B_2(T) = b - \frac{a}{RT}, \quad B_3(T) = b^2
$$

$$
\boxed{ B_2(T) = b - \frac{a}{RT}, \quad B_3(T) = b^2 }
$$

**Key takeaway**
The second virial coefficient $B_2(T)$ reflects the competition between repulsive volume exclusion ($b$) and attractive forces ($-a/RT$).

In [25]:
# L2.6 — the virial coefficients B_2 = b - a/(RT) and B_3 = b^2 read off numerically.
R, T, a_vdw, b_vdw = 8.314, 300.0, 0.1408, 3.913e-5      # nitrogen, SI molar units
P = lambda rho: R * T * rho / (1 - b_vdw * rho) - a_vdw * rho ** 2
B2_pred = b_vdw - a_vdw / (R * T)
B3_pred = b_vdw ** 2

# fit P/(RT) = rho + B2 rho^2 + B3 rho^3 + ... on small densities
rho = np.linspace(1e-3, 5.0, 400)
y = (P(rho) / (R * T) - rho) / rho ** 2                   # -> B2 + B3 rho as rho -> 0
coef = np.polyfit(rho, y, 1)
print(f"B_2 fitted = {coef[1]:.8e}    predicted b - a/(RT) = {B2_pred:.8e}")
print(f"B_3 fitted = {coef[0]:.8e}    predicted b^2        = {B3_pred:.8e}")
assert abs(coef[1] - B2_pred) / abs(B2_pred) < 1e-3
assert abs(coef[0] - B3_pred) / abs(B3_pred) < 5e-2
print(f"\nB_2 changes sign at the Boyle temperature T_B = a/(Rb) = {a_vdw/(R*b_vdw):.1f} K")

B_2 fitted = -1.73209665e-05    predicted b - a/(RT) = -1.73209662e-05
B_3 fitted = 1.53145735e-09    predicted b^2        = 1.53115690e-09

B_2 changes sign at the Boyle temperature T_B = a/(Rb) = 432.8 K


### Problem L2.7 — AI/ML — Softplus Asymptotics and the ReLU Limit
**Source:** Goodfellow et al., *Deep Learning*, Ch. 6

**Statement**

Analyze the asymptotic expansion of the Softplus function $f(x) = \ln(1 + e^x)$ for:  
(a) $x \to +\infty$  
(b) $x \to -\infty$  
Show how $f(x)$ approaches $\operatorname{ReLU}(x) = \max(0, x)$ in both regimes.

**Intuition**
For large positive $x$, $e^x \gg 1$, so factor out $e^x$. For large negative $x$, $e^x \ll 1$, so use $\ln(1+u) \approx u$.

**Solution**
(a) **Regime $x \to +\infty$**:

$$
f(x) = \ln(e^x(1 + e^{-x})) = \ln(e^x) + \ln(1 + e^{-x}) = x + \ln(1 + e^{-x})
$$

Since $e^{-x} \to 0$ as $x \to +\infty$, apply Maclaurin expansion of $\ln(1+u)$ for $u = e^{-x}$:

$$
f(x) = x + e^{-x} - \frac{1}{2}e^{-2x} + O(e^{-3x})
$$

As $x \to +\infty$, $f(x) \to x = \operatorname{ReLU}(x)$ with exponentially decaying error $O(e^{-x})$.

(b) **Regime $x \to -\infty$**:
As $x \to -\infty$, $u = e^x \to 0$.  
Apply Maclaurin expansion of $\ln(1+u)$ directly:

$$
f(x) = e^x - \frac{1}{2}e^{2x} + \frac{1}{3}e^{3x} + O(e^{4x})
$$

As $x \to -\infty$, $f(x) \to 0 = \operatorname{ReLU}(x)$ with exponentially decaying error $O(e^x)$.

$$
\boxed{ f(x) = \begin{cases} x + e^{-x} + O(e^{-2x}) & \text{as } x \to +\infty \\ e^x - \frac{1}{2}e^{2x} + O(e^{3x}) & \text{as } x \to -\infty \end{cases} }
$$

**Key takeaway**
Softplus is a $C^\infty$ smooth approximation to ReLU whose asymptotic residuals decay exponentially $O(e^{-\lvert x \rvert})$.

In [26]:
# L2.7 — softplus approaches ReLU with exponentially small residuals in both tails.
softplus = lambda x: np.logaddexp(0.0, x)
relu = lambda x: np.maximum(0.0, x)
print(f"{'x':>7s} {'softplus':>16s} {'ReLU':>10s} {'residual':>14s} {'e^{-|x|}':>12s}")
for x in (-20.0, -10.0, -5.0, 5.0, 10.0, 20.0):
    res = softplus(x) - relu(x)
    print(f"{x:7.1f} {softplus(x):16.12f} {relu(x):10.4f} {res:14.6e} {np.exp(-abs(x)):12.3e}")
    assert 0 < res <= np.exp(-abs(x))
# the two-term expansions
for x in (8.0, 12.0):
    print(f"  x = {x}: x + e^-x - e^-2x/2 = {x + np.exp(-x) - np.exp(-2*x)/2:.14f}"
          f"   exact = {softplus(x):.14f}")
    assert abs(softplus(x) - (x + np.exp(-x) - np.exp(-2*x)/2)) < np.exp(-3*x)
for x in (-8.0, -12.0):
    print(f"  x = {x}: e^x - e^2x/2 + e^3x/3 = {np.exp(x) - np.exp(2*x)/2 + np.exp(3*x)/3:.14f}"
          f"   exact = {softplus(x):.14f}")
print("\nverified: |softplus(x) - ReLU(x)| <= e^{-|x|} in both regimes")

      x         softplus       ReLU       residual     e^{-|x|}
  -20.0   0.000000002061     0.0000   2.061154e-09    2.061e-09
  -10.0   0.000045398899     0.0000   4.539890e-05    4.540e-05
   -5.0   0.006715348489     0.0000   6.715348e-03    6.738e-03
    5.0   5.006715348489     5.0000   6.715348e-03    6.738e-03
   10.0  10.000045398899    10.0000   4.539890e-05    4.540e-05
   20.0  20.000000002061    20.0000   2.061153e-09    2.061e-09
  x = 8.0: x + e^-x - e^-2x/2 = 8.00033540636031   exact = 8.00033540637290
  x = 12.0: x + e^-x - e^-2x/2 = 12.00000614419348   exact = 12.00000614419348
  x = -8.0: e^x - e^2x/2 + e^3x/3 = 0.00033540637290   exact = 0.00033540637290
  x = -12.0: e^x - e^2x/2 + e^3x/3 = 0.00000614419348   exact = 0.00000614419348

verified: |softplus(x) - ReLU(x)| <= e^{-|x|} in both regimes


### Problem L2.8 — Physics — Electric Dipole Potential
**Source:** Griffiths, *Introduction to Electrodynamics*, Ch. 3

**Statement**

Two opposite point charges $+q$ and $-q$ are located at $z = +d/2$ and $z = -d/2$.  
Derive the leading non-zero term in the Taylor expansion of the potential $V(r, \theta)$ at a distance $r \gg d$.

**Intuition**
The monopole terms cancel ($q - q = 0$). The leading non-zero contribution is the dipole term proportional to $\frac{1}{r^2}$.

**Solution**
The potential at point $P(r, \theta)$ is:

$$
V = \frac{q}{4\pi\varepsilon_0} \left( \frac{1}{\mathcal{R}_+} - \frac{1}{\mathcal{R}_-} \right)
$$

By the Law of Cosines:

$$
\mathcal{R}_\pm = \sqrt{r^2 \mp r d \cos\theta + \frac{d^2}{4}} = r \sqrt{1 \mp \frac{d}{r}\cos\theta + \frac{d^2}{4r^2}}
$$

Let $x = \frac{d}{r}\cos\theta - \frac{d^2}{4r^2}$. Expand $(1 - x)^{-\frac{1}{2}} = 1 + \frac{1}{2}x + O(x^2)$:

$$
\frac{1}{\mathcal{R}_+} \approx \frac{1}{r} \left( 1 + \frac{d}{2r}\cos\theta \right), \quad \frac{1}{\mathcal{R}_-} \approx \frac{1}{r} \left( 1 - \frac{d}{2r}\cos\theta \right)
$$

Subtract the two distances:

$$
\frac{1}{\mathcal{R}_+} - \frac{1}{\mathcal{R}_-} \approx \frac{1}{r} \left( \frac{d}{r}\cos\theta \right) = \frac{d \cos\theta}{r^2}
$$

Substitute back into $V$:

$$
V(r, \theta) \approx \frac{q d \cos\theta}{4\pi\varepsilon_0 r^2} = \frac{p \cos\theta}{4\pi\varepsilon_0 r^2} \quad \text{where } p = qd
$$

$$
\boxed{ V(r, \theta) = \frac{p \cos\theta}{4\pi\varepsilon_0 r^2} + O\left(\frac{1}{r^3}\right) }
$$

**Key takeaway**
Multipole expansions in physics are spatial Taylor expansions in the small parameter $d/r$.

In [27]:
# L2.8 — the dipole potential: exact two-charge sum against the p cos(theta) / r^2 model.
q, d = 1.0, 1.0
k_e = 1.0                                     # work in units where 1/(4 pi eps_0) = 1
def V_exact(r, theta):
    Rp = np.sqrt(r ** 2 - r * d * np.cos(theta) + d ** 2 / 4)
    Rm = np.sqrt(r ** 2 + r * d * np.cos(theta) + d ** 2 / 4)
    return k_e * q * (1 / Rp - 1 / Rm)

V_dip = lambda r, theta: k_e * q * d * np.cos(theta) / r ** 2
theta = 0.7
print(f"{'r/d':>8s} {'V exact':>18s} {'dipole model':>18s} {'|rel. error|':>14s} {'(d/r)^2':>12s}")
for r in (5.0, 10.0, 50.0, 200.0):
    ve, vd = V_exact(r, theta), V_dip(r, theta)
    rel = abs(ve - vd) / abs(ve)
    print(f"{r:8.1f} {ve:18.12e} {vd:18.12e} {rel:14.3e} {(d/r)**2:12.3e}")
    assert rel < 2.0 * (d / r) ** 2
print("\nthe relative error falls like (d/r)^2: the next multipole is the octupole, O(1/r^4)")

     r/d            V exact       dipole model   |rel. error|      (d/r)^2
     5.0 3.058051980257e-02 3.059368749138e-02      4.306e-04    4.000e-02
    10.0 7.647677802131e-03 7.648421872845e-03      9.729e-05    1.000e-02
    50.0 3.059357247152e-04 3.059368749138e-04      3.760e-06    4.000e-04
   200.0 1.912105019530e-05 1.912105468211e-05      2.347e-07    2.500e-05

the relative error falls like (d/r)^2: the next multipole is the octupole, O(1/r^4)


### Problem L2.9 — AI/ML — KL Divergence and Fisher Information
**Source:** Amari, *Information Geometry and Its Applications*

**Statement**

Let $P_\theta$ be a parametrized probability distribution. Derive the 2nd-order Taylor expansion of the Kullback-Leibler divergence $D_{KL}(P_\theta \,\lVert\, P_{\theta + \Delta \theta})$ for small perturbations $\Delta \theta$.

**Intuition**
$D_{KL}(P_\theta \| P_\theta) = 0$ is a global minimum. Thus the 0th and 1st order Taylor terms vanish, making the leading term pure quadratic $\frac{1}{2}\Delta\theta^T F \Delta\theta$.

**Solution**
1. **Definition**:

$$
   D_{KL}(P_\theta \,\lVert\, P_{\theta + \Delta \theta}) = \int p(x; \theta) \ln \frac{p(x; \theta)}{p(x; \theta + \Delta \theta)} \, dx
$$

2. **Taylor Expand $\ln p(x; \theta + \Delta \theta)$**:

$$
   \ln p(x; \theta + \Delta \theta) = \ln p(x; \theta) + \Delta \theta^T \nabla_\theta \ln p(x; \theta) + \frac{1}{2} \Delta \theta^T \left[ \nabla_\theta^2 \ln p(x; \theta) \right] \Delta \theta + O(\lVert \Delta \theta \rVert^3)
$$

3. **Substitute into KL Divergence**:

$$
   D_{KL} = \int p(x; \theta) \left[ -\Delta \theta^T \nabla_\theta \ln p(x; \theta) - \frac{1}{2} \Delta \theta^T \nabla_\theta^2 \ln p(x; \theta) \Delta \theta \right] dx
$$

4. **Evaluate Expectations**:
   - First term: $\mathbb{E}_\theta [\nabla_\theta \ln p(x; \theta)] = \int \nabla_\theta p(x; \theta) dx = \nabla_\theta (1) = 0$.
   - Second term: $-\mathbb{E}_\theta [\nabla_\theta^2 \ln p(x; \theta)] = F(\theta)$, the **Fisher Information Matrix**.

Thus:

$$
D_{KL}(P_\theta \,\lVert\, P_{\theta + \Delta \theta}) = \frac{1}{2} \Delta \theta^T F(\theta) \Delta \theta + O(\lVert \Delta \theta \rVert^3)
$$

$$
\boxed{ D_{KL}(P_\theta \,\lVert\, P_{\theta + \Delta \theta}) = \frac{1}{2} \Delta \theta^T F(\theta) \Delta \theta + O(\lVert \Delta \theta \rVert^3) }
$$

**Key takeaway**
The Fisher Information Matrix is the Riemannian metric tensor generated by the 2nd-order Taylor expansion of KL divergence.

In [28]:
# L2.9 — KL divergence between two Gaussians is (1/2) F dtheta^2 to leading order.
# For N(mu, 1) with theta = mu, the Fisher information is F = 1 and D_KL = dmu^2 / 2 exactly.
# Take instead theta = (mu, log sigma) so the quadratic form has two components.
def kl_gauss(m0, s0, m1, s1):
    return np.log(s1 / s0) + (s0 ** 2 + (m0 - m1) ** 2) / (2 * s1 ** 2) - 0.5

mu, sig = 0.3, 1.4
F = np.array([[1 / sig ** 2, 0.0], [0.0, 2.0]])      # Fisher matrix in (mu, log sigma)
print(f"{'|dtheta|':>10s} {'D_KL exact':>18s} {'(1/2) dth^T F dth':>20s} {'ratio':>10s}")
direction = np.array([0.6, -0.8])
for eps in (1e-1, 1e-2, 1e-3):
    dth = eps * direction
    exact = kl_gauss(mu, sig, mu + dth[0], sig * np.exp(dth[1]))
    quad_form = 0.5 * dth @ F @ dth
    print(f"{eps:10.0e} {exact:18.12e} {quad_form:20.12e} {exact/quad_form:10.6f}")
    assert abs(exact / quad_form - 1.0) < 2.0 * eps
print("\nthe ratio -> 1 as the perturbation shrinks: F is the local metric of the KL divergence")

  |dtheta|         D_KL exact    (1/2) dth^T F dth      ratio
     1e-01 7.833149561102e-03   7.318367346939e-03   1.070341
     1e-02 7.367449709694e-05   7.318367346939e-05   1.006707
     1e-03 7.323252610991e-07   7.318367346939e-07   1.000668

the ratio -> 1 as the perturbation shrinks: F is the local metric of the KL divergence


### Problem L2.10 — Physics — Rayleigh-Jeans Limit of the Planck Spectrum
**Source:** Statistical Mechanics / Quantum Physics

**Statement**

Planck's spectral energy density is $u(\nu) = \frac{8\pi h \nu^3}{c^3} \frac{1}{e^{\frac{h\nu}{kT}} - 1}$.  
Show that for low frequencies $h\nu \ll kT$, the leading Taylor term recovers the classical Rayleigh-Jeans law.

**Intuition**
For $x = \frac{h\nu}{kT} \ll 1$, expand $e^x - 1 \approx x$.

**Solution**
Let $x = \frac{h\nu}{kT} \ll 1$. Taylor expand $e^x$:

$$
e^x = 1 + x + \frac{x^2}{2} + O(x^3) \implies e^x - 1 = x \left( 1 + \frac{x}{2} + O(x^2) \right)
$$

Take the reciprocal:

$$
\frac{1}{e^x - 1} = \frac{1}{x} \left( 1 + \frac{x}{2} \right)^{-1} = \frac{1}{x} \left( 1 - \frac{x}{2} + O(x^2) \right) = \frac{1}{x} - \frac{1}{2} + O(x)
$$

Substitute into $u(\nu)$:

$$
\begin{aligned} u(\nu) &= \frac{8\pi h \nu^3}{c^3} \left( \frac{kT}{h\nu} - \frac{1}{2} + O(\nu) \right) \\ &= \frac{8\pi \nu^2 k T}{c^3} - \frac{4\pi h \nu^3}{c^3} + O(\nu^4) \end{aligned}
$$

The leading term is:

$$
u_{RJ}(\nu) = \frac{8\pi \nu^2 k T}{c^3}
$$

$$
\boxed{ u(\nu) = \frac{8\pi \nu^2 k T}{c^3} - \frac{4\pi h \nu^3}{c^3} + O(\nu^4) }
$$

**Key takeaway**
Quantum radiation theory smoothly reduces to classical equipartition thermodynamics via 1st-order Taylor expansion.

In [29]:
# L2.10 — the Planck spectrum reduces to Rayleigh-Jeans at low frequency.
h, kB, c_light, T = 6.62607015e-34, 1.380649e-23, 2.99792458e8, 300.0
def u_planck(nu):
    x = h * nu / (kB * T)
    return 8 * np.pi * h * nu ** 3 / c_light ** 3 / np.expm1(x)

u_rj = lambda nu: 8 * np.pi * nu ** 2 * kB * T / c_light ** 3
u_two = lambda nu: u_rj(nu) - 4 * np.pi * h * nu ** 3 / c_light ** 3
print(f"{'h nu / kT':>12s} {'Planck':>16s} {'Rayleigh-Jeans':>18s} {'two-term':>18s} {'RJ rel. err':>13s}")
for x in (1e-3, 1e-2, 1e-1, 1.0):
    nu = x * kB * T / h
    p, r, t2 = u_planck(nu), u_rj(nu), u_two(nu)
    print(f"{x:12.0e} {p:16.6e} {r:18.6e} {t2:18.6e} {abs(r-p)/p:13.3e}")
    if x <= 0.1:
        assert abs(t2 - p) / p < abs(r - p) / p
nu = 1e-2 * kB * T / h
assert abs(u_rj(nu) - u_planck(nu)) / u_planck(nu) < 6e-3
print("\nthe leading term is Rayleigh-Jeans; the -4 pi h nu^3 / c^3 term is the first quantum correction")

   h nu / kT           Planck     Rayleigh-Jeans           two-term   RJ rel. err
       1e-03     1.508906e-25       1.509661e-25       1.508906e-25     5.002e-04
       1e-02     1.502125e-23       1.509661e-23       1.502112e-23     5.017e-03
       1e-01     1.435435e-21       1.509661e-21       1.434178e-21     5.171e-02
       1e+00     8.785873e-20       1.509661e-19       7.548303e-20     7.183e-01

the leading term is Rayleigh-Jeans; the -4 pi h nu^3 / c^3 term is the first quantum correction


### Problem L2.11 — AI/ML — SiLU / Swish Expansion
**Source:** Elfwing et al. (2018), *Sigmoid-Weighted Linear Units*

**Statement**

The SiLU activation function is $f(x) = \frac{x}{1 + e^{-x}} = x \sigma(x)$.  
Derive its Maclaurin polynomial through the $x^4$ term.

**Intuition**
Multiply $x$ by the Maclaurin expansion of $\sigma(x) = \frac{1}{2} + \frac{x}{4} - \frac{x^3}{48} + \dots$

**Solution**
1. **Expansion of Sigmoid $\sigma(x)$**:

$$
   \sigma(x) = \frac{1}{1+e^{-x}} = \frac{1}{2} + \frac{1}{2}\tanh(\frac{x}{2})
$$

   Using $\tanh(u) = u - \frac{u^3}{3} + O(u^5)$:

$$
   \sigma(x) = \frac{1}{2} + \frac{1}{2}\left( \frac{x}{2} - \frac{x^3}{24} \right) + O(x^5) = \frac{1}{2} + \frac{x}{4} - \frac{x^3}{48} + O(x^5)
$$

2. **Multiply by $x$**:

$$
   f(x) = x \left( \frac{1}{2} + \frac{x}{4} - \frac{x^3}{48} + O(x^5) \right) = \frac{1}{2}x + \frac{1}{4}x^2 - \frac{1}{48}x^4 + O(x^6)
$$

$$
\boxed{ f(x) = \frac{1}{2}x + \frac{1}{4}x^2 - \frac{1}{48}x^4 + O(x^6) }
$$

**Key takeaway**
Near $x=0$, $f'(x) = \frac{1}{2} + \frac{1}{2}x - \frac{1}{12}x^3 + O(x^5)$ is strictly positive, so SiLU is monotone increasing at the origin; its true non-monotonicity is a global feature, with the unique local minimum near $x \approx -1.28$, far outside the radius where this quartic Maclaurin term is a good approximation.

In [30]:
# L2.11 — the SiLU expansion x/2 + x^2/4 - x^4/48, and the true minimum near x = -1.2785.
from scipy.optimize import brentq

silu = lambda x: x / (1 + np.exp(-x))
approx = lambda x: x / 2 + x ** 2 / 4 - x ** 4 / 48
print(f"{'x':>7s} {'SiLU exact':>16s} {'quartic model':>16s} {'|error|':>12s} {'|error|/x^6':>14s}")
for x in (0.4, 0.2, 0.1, 0.05):
    e = abs(silu(x) - approx(x))
    print(f"{x:7.2f} {silu(x):16.12f} {approx(x):16.12f} {e:12.3e} {e/x**6:14.6f}")
    assert e < 0.01 * x ** 6 + 1e-15
dsilu = lambda x: (1 + np.exp(-x) + x * np.exp(-x)) / (1 + np.exp(-x)) ** 2
xmin = brentq(dsilu, -3.0, -0.5)
print(f"\nSiLU'(0) = {dsilu(0.0):.6f}  (model predicts 1/2)")
print(f"unique local minimum at x = {xmin:.6f}, value {silu(xmin):.6f}")
assert abs(dsilu(0.0) - 0.5) < 1e-12
assert abs(xmin + 1.278465) < 1e-4
print("the minimum sits far outside the range where the quartic model is accurate")

      x       SiLU exact    quartic model      |error|    |error|/x^6
   0.40   0.239475064045   0.239466666667    8.397e-06       0.002050
   0.20   0.109966799462   0.109966666667    1.328e-07       0.002075
   0.10   0.052497918748   0.052497916667    2.081e-09       0.002081
   0.05   0.025624869824   0.025624869792    3.254e-11       0.002083

SiLU'(0) = 0.500000  (model predicts 1/2)
unique local minimum at x = -1.278465, value -0.278465
the minimum sits far outside the range where the quartic model is accurate


### Problem L2.12 — Numerical Analysis — Optimal Central-Difference Step
**Source:** Burden & Faires, *Numerical Analysis*, Ch. 4

**Statement**

The central difference approximation for $f'(x)$ is $D_h = \frac{f(x+h) - f(x-h)}{2h}$.  
(a) Derive the truncation error using Taylor series.  
(b) Including floating-point roundoff error bounded by $\frac{\varepsilon_{mach}}{h}$, find the optimal step size $h^\ast$ that minimizes total error.

**Intuition**
Truncation error scales as $O(h^2)$ (decreases as $h \to 0$). Roundoff error scales as $O(\varepsilon/h)$ (increases as $h \to 0$). Total error has a global minimum $h^\ast$.

**Solution**
(a) Taylor expansions:

$$
f(x+h) = f(x) + h f'(x) + \frac{h^2}{2} f''(x) + \frac{h^3}{6} f'''(x) + O(h^4)
$$

$$
f(x-h) = f(x) - h f'(x) + \frac{h^2}{2} f''(x) - \frac{h^3}{6} f'''(x) + O(h^4)
$$

Subtract:

$$
f(x+h) - f(x-h) = 2h f'(x) + \frac{h^3}{3} f'''(x) + O(h^5)
$$

Divide by $2h$:

$$
D_h = f'(x) + \frac{h^2}{6} f'''(x) + O(h^4) \implies E_{trunc} = \frac{M_3}{6} h^2
$$

(b) Total Error $E(h) = \frac{M_3}{6} h^2 + \frac{\varepsilon_{mach}}{h}$.  
Differentiate with respect to $h$ and set to zero:

$$
E'(h) = \frac{M_3}{3} h - \frac{\varepsilon_{mach}}{h^2} = 0 \implies h^3 = \frac{3 \varepsilon_{mach}}{M_3}
$$

$$
h^\ast = \left( \frac{3 \varepsilon_{mach}}{M_3} \right)^{\frac{1}{3}} = O(\varepsilon_{mach}^{\frac{1}{3}})
$$

$$
\boxed{ E_{trunc} = \frac{h^2}{6} f'''(x), \quad h^\ast = \left( \frac{3 \varepsilon_{mach}}{M_3} \right)^{\frac{1}{3}} }
$$

**Key takeaway**
Taylor truncation error drives optimal numerical step size design in scientific computation.

In [31]:
# L2.12 — the optimal central-difference step h* = (3 eps / M_3)^(1/3), measured on sin.
# Round-off at one base point is erratic, so the error is averaged (RMS) over many base
# points near x0; that is what makes the measured minimiser reproducible.
EPS = np.finfo(float).eps
f, x0 = np.sin, 1.0
hs = np.logspace(-12.0, -1.0, 220)
xs = np.linspace(x0 - 0.2, x0 + 0.2, 401)
D = (f(xs[None, :] + hs[:, None]) - f(xs[None, :] - hs[:, None])) / (2 * hs[:, None])
errs = np.sqrt(np.mean((D - np.cos(xs)[None, :]) ** 2, axis=1))
h_meas = hs[int(np.argmin(errs))]
M3 = abs(np.sin(x0))                                   # third derivative magnitude of sin at x0
h_pred = (3 * EPS / M3) ** (1 / 3)
print(f"measured optimal h            = {h_meas:.4e}")
print(f"predicted (3 eps / M_3)^(1/3) = {h_pred:.4e}")
print(f"ratio measured / predicted    = {h_meas/h_pred:.4f}")
print(f"smallest achievable RMS error = {errs.min():.4e}")
print(f"error at h = 1e-10            = "
      f"{abs((f(x0+1e-10)-f(x0-1e-10))/2e-10 - np.cos(x0)):.4e}   (round-off dominated)")
# and the O(h^2) truncation order in the regime where round-off is negligible
mid = (hs > 1e-4) & (hs < 1e-2)
slope = np.polyfit(np.log(hs[mid]), np.log(errs[mid]), 1)[0]
print(f"\nfitted truncation order on [1e-4, 1e-2] : {slope:.4f}   (predicted 2)")
assert 0.3 < h_meas / h_pred < 3.0
assert abs(slope - 2.0) < 0.05

measured optimal h            = 6.7772e-06
predicted (3 eps / M_3)^(1/3) = 9.2507e-06
ratio measured / predicted    = 0.7326
smallest achievable RMS error = 4.3426e-12
error at h = 1e-10            = 5.8481e-08   (round-off dominated)

fitted truncation order on [1e-4, 1e-2] : 2.0000   (predicted 2)


## L3 — Challenge Proofs

Ten proofs and asymptotic analyses at competition and graduate-course level.

### Problem L3.1 — Even Sub-Series of the Exponential
**Source:** Standard exercise (see e.g. Rudin, *Principles of Mathematical Analysis*, Ch. 8, on the exponential series); not from a named competition.

**Statement**

Find a closed form for the infinite series:

$$
S(x) = \sum_{n=0}^\infty \frac{x^{2n}}{(2n)!}
$$

and use it to evaluate $\sum_{n=0}^\infty \frac{1}{(2n)!}$.

**Intuition**
Notice that $\frac{x^{2n}}{(2n)!}$ contains only even power terms of $e^x = \sum \frac{x^n}{n!}$. Averaging $e^x$ and $e^{-x}$ isolates even powers.

**Solution**
Recall the Maclaurin series for $e^x$ and $e^{-x}$:

$$
e^x = \sum_{k=0}^\infty \frac{x^k}{k!} = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \dots
$$

$$
e^{-x} = \sum_{k=0}^\infty \frac{(-x)^k}{k!} = 1 - x + \frac{x^2}{2!} - \frac{x^3}{3!} + \dots
$$

Add the two series:

$$
e^x + e^{-x} = 2 \left( 1 + \frac{x^2}{2!} + \frac{x^4}{4!} + \dots \right) = 2 \sum_{n=0}^\infty \frac{x^{2n}}{(2n)!}
$$

Divide by 2:

$$
S(x) = \sum_{n=0}^\infty \frac{x^{2n}}{(2n)!} = \frac{e^x + e^{-x}}{2} = \cosh(x)
$$

Setting $x = 1$:

$$
\sum_{n=0}^\infty \frac{1}{(2n)!} = \cosh(1) = \frac{e + e^{-1}}{2}
$$

$$
\boxed{ S(x) = \cosh(x), \quad \sum_{n=0}^\infty \frac{1}{(2n)!} = \frac{e + e^{-1}}{2} }
$$

**Key takeaway**
Symmetric combination of series $f(x) \pm f(-x)$ extracts even and odd component subseries.

In [32]:
# L3.1 — sum x^(2n)/(2n)! = cosh x, and its value at x = 1.
ns = np.arange(0, 40)
logfact2 = np.array([sum(np.log(j) for j in range(1, 2 * int(n) + 1)) for n in ns])
print(f"{'x':>6s} {'series':>20s} {'cosh(x)':>20s} {'|difference|':>14s}")
for x in (0.5, 1.0, 2.0, 3.0):
    s = float(np.sum(x ** (2 * ns) * np.exp(-logfact2)))
    print(f"{x:6.2f} {s:20.14f} {np.cosh(x):20.14f} {abs(s-np.cosh(x)):14.3e}")
    assert abs(s - np.cosh(x)) < 1e-12
s1 = float(np.sum(np.exp(-logfact2)))
print(f"\nsum 1/(2n)! = {s1:.14f}   cosh(1) = {np.cosh(1.0):.14f}   (e + 1/e)/2 = "
      f"{(np.e + 1/np.e)/2:.14f}")
assert abs(s1 - (np.e + 1 / np.e) / 2) < 1e-14

     x               series              cosh(x)   |difference|
  0.50     1.12762596520638     1.12762596520638      0.000e+00
  1.00     1.54308063481524     1.54308063481524      2.220e-16
  2.00     3.76219569108363     3.76219569108363      4.441e-16
  3.00    10.06766199577777    10.06766199577777      0.000e+00

sum 1/(2n)! = 1.54308063481524   cosh(1) = 1.54308063481524   (e + 1/e)/2 = 1.54308063481524


### Problem L3.2 — Multisection by Fourth Roots of Unity
**Source:** Demidovich, *Problems in Analysis*, No. 2542

**Statement**

Evaluate the closed-form sum of the power series:

$$
f(x) = \sum_{n=0}^\infty \frac{x^{4n}}{(4n)!}
$$

**Intuition**
This requires multi-sectioning using the 4-th roots of unity $\pm 1, \pm i$. Summing $e^z$ evaluated at all 4 roots isolates terms where power is a multiple of 4.

**Solution**
The 4th roots of unity are $w_k \in \{1, i, -1, -i\}$.  
Recall identity:

$$
\frac{1}{4} \sum_{k=0}^3 w_k^m = \begin{cases} 1 & \text{if } m \equiv 0 \pmod 4 \\ 0 & \text{otherwise} \end{cases}
$$

Evaluate $e^{w_k x}$:

$$
\begin{aligned} S &= \frac{1}{4} \left[ e^x + e^{-x} + e^{ix} + e^{-ix} \right] \\ &= \frac{1}{4} \left[ (e^x + e^{-x}) + (e^{ix} + e^{-ix}) \right] \\ &= \frac{1}{4} \left[ 2\cosh(x) + 2\cos(x) \right] = \frac{\cosh(x) + \cos(x)}{2} \end{aligned}
$$

Expanding $S$:

$$
\frac{\cosh(x) + \cos(x)}{2} = \frac{1}{2} \left( \sum_{n=0}^\infty \frac{x^{2n}}{(2n)!} + \sum_{n=0}^\infty \frac{(-1)^n x^{2n}}{(2n)!} \right) = \sum_{m=0}^\infty \frac{x^{4m}}{(4m)!}
$$

$$
\boxed{ \sum_{n=0}^\infty \frac{x^{4n}}{(4n)!} = \frac{\cosh(x) + \cos(x)}{2} }
$$

**Key takeaway**
Roots of unity filter specific congruence classes of coefficients in power series (series multisectioning).

In [33]:
# L3.2 — the 4-section: sum x^(4n)/(4n)! = (cosh x + cos x) / 2.
ns = np.arange(0, 25)
logfact4 = np.array([sum(np.log(j) for j in range(1, 4 * int(n) + 1)) for n in ns])
print(f"{'x':>6s} {'series':>20s} {'(cosh x + cos x)/2':>22s} {'|difference|':>14s}")
for x in (0.5, 1.0, 2.0, 4.0):
    s = float(np.sum(x ** (4 * ns) * np.exp(-logfact4)))
    closed = (np.cosh(x) + np.cos(x)) / 2
    print(f"{x:6.2f} {s:20.13f} {closed:22.13f} {abs(s-closed):14.3e}")
    assert abs(s - closed) / max(1.0, abs(closed)) < 1e-12
# the root-of-unity filter itself
for m in range(9):
    filt = sum(w ** m for w in (1, 1j, -1, -1j)) / 4
    print(f"  m = {m}: (1/4) sum w^m = {filt.real:+.0f}   "
          f"{'(m divisible by 4)' if m % 4 == 0 else ''}")
    assert abs(filt - (1.0 if m % 4 == 0 else 0.0)) < 1e-12

     x               series     (cosh x + cos x)/2   |difference|
  0.50      1.0026042635484        1.0026042635484      0.000e+00
  1.00      1.0416914703417        1.0416914703417      0.000e+00
  2.00      1.6730244272682        1.6730244272682      4.441e-16
  4.00     13.3272946075764       13.3272946075764      1.776e-15
  m = 0: (1/4) sum w^m = +1   (m divisible by 4)
  m = 1: (1/4) sum w^m = +0   
  m = 2: (1/4) sum w^m = +0   
  m = 3: (1/4) sum w^m = +0   
  m = 4: (1/4) sum w^m = +1   (m divisible by 4)
  m = 5: (1/4) sum w^m = +0   
  m = 6: (1/4) sum w^m = +0   
  m = 7: (1/4) sum w^m = +0   
  m = 8: (1/4) sum w^m = +1   (m divisible by 4)


### Problem L3.3 — Rate of Convergence of $(1+1/n)^n$
**Source:** Cambridge Mathematical Tripos, Part IA

**Statement**

Compute the exact value of the limit:

$$
L = \lim_{n\to\infty} n \left( \left( 1 + \frac{1}{n} \right)^n - e \right)
$$

**Intuition**
Use Taylor expansion of $\ln(1+x)$ to expand $\left(1+\frac{1}{n}\right)^n = \exp\left( n \ln\left(1+\frac{1}{n}\right) \right)$ up to $O(1/n^2)$.

**Solution**
1. **Expand Logarithm**:

$$
   \ln\left(1 + \frac{1}{n}\right) = \frac{1}{n} - \frac{1}{2n^2} + O\left(\frac{1}{n^3}\right)
$$

2. **Multiply by $n$**:

$$
   n \ln\left(1 + \frac{1}{n}\right) = 1 - \frac{1}{2n} + O\left(\frac{1}{n^2}\right)
$$

3. **Exponentiate using $e^u = e \cdot e^{u-1}$**:

$$
   \left(1 + \frac{1}{n}\right)^n = \exp\left( 1 - \frac{1}{2n} + O(1/n^2) \right) = e \cdot \exp\left( -\frac{1}{2n} + O(1/n^2) \right)
$$

4. **Expand $\exp(y)$ for $y = -\frac{1}{2n}$**:

$$
   \exp\left( -\frac{1}{2n} + O(1/n^2) \right) = 1 - \frac{1}{2n} + O\left(\frac{1}{n^2}\right)
$$

   Thus:

$$
   \left(1 + \frac{1}{n}\right)^n = e \left( 1 - \frac{1}{2n} + O(1/n^2) \right) = e - \frac{e}{2n} + O\left(\frac{1}{n^2}\right)
$$

5. **Substitute into Limit**:

$$
   L = \lim_{n\to\infty} n \left( e - \frac{e}{2n} + O(1/n^2) - e \right) = \lim_{n\to\infty} \left( -\frac{e}{2} + O(\frac{1}{n}) \right) = -\frac{e}{2}
$$

$$
\boxed{ L = -\frac{e}{2} }
$$

**Key takeaway**
Asymptotic expansion via log-exp transformation unveils rates of limit convergence.

In [34]:
# L3.3 — n((1 + 1/n)^n - e) -> -e/2.
print(f"{'n':>10s} {'n((1+1/n)^n - e)':>22s} {'-e/2':>14s} {'difference':>14s}")
for n in (10, 10 ** 2, 10 ** 3, 10 ** 4, 10 ** 5):
    val = n * (np.exp(n * np.log1p(1.0 / n)) - np.e)
    print(f"{n:10d} {val:22.12f} {-np.e/2:14.10f} {val + np.e/2:14.3e}")
target = 10 ** 5 * (np.exp(10 ** 5 * np.log1p(1e-5)) - np.e)
assert abs(target + np.e / 2) < 1e-3
import sympy as sp
ns = sp.symbols('n', positive=True)
print("\nsympy limit :", sp.limit(ns * ((1 + 1 / ns) ** ns - sp.E), ns, sp.oo))
assert sp.limit(ns * ((1 + 1 / ns) ** ns - sp.E), ns, sp.oo) == -sp.E / 2

         n       n((1+1/n)^n - e)           -e/2     difference
        10        -1.245393683590  -1.3591409142      1.137e-01
       100        -1.346799903752  -1.3591409142      1.234e-02
      1000        -1.357896223153  -1.3591409142      1.245e-03
     10000        -1.359016338203  -1.3591409142      1.246e-04
    100000        -1.359128455514  -1.3591409142      1.246e-05



sympy limit : -E/2


### Problem L3.4 — Perturbation Expansion of an Algebraic Root
**Source:** Bender & Orszag, *Advanced Mathematical Methods*, Ch. 1

**Statement**

Find a 2nd-order perturbation expansion in $\varepsilon$ for the root of $x + \varepsilon x^5 = 1$ near $x=1$ as $\varepsilon \to 0$.

**Intuition**
Assume a power series expansion $x(\varepsilon) = x_0 + \varepsilon x_1 + \varepsilon^2 x_2 + O(\varepsilon^3)$. Substitute into the equation and equate coefficients of $\varepsilon^k$.

**Solution**
1. **Unperturbed problem ($\varepsilon = 0$)**:

$$
   x_0 = 1
$$

2. **Ansatz**:

$$
   x(\varepsilon) = 1 + \varepsilon x_1 + \varepsilon^2 x_2 + O(\varepsilon^3)
$$

3. **Expand $x^5$**:

$$
   x^5 = (1 + \varepsilon x_1 + \varepsilon^2 x_2)^5 = 1 + 5\varepsilon x_1 + O(\varepsilon^2)
$$

4. **Substitute into $x + \varepsilon x^5 = 1$**:

$$
   (1 + \varepsilon x_1 + \varepsilon^2 x_2) + \varepsilon (1 + 5\varepsilon x_1 + O(\varepsilon^2)) = 1
$$

$$
   1 + \varepsilon(x_1 + 1) + \varepsilon^2(x_2 + 5x_1) + O(\varepsilon^3) = 1
$$

5. **Equate Coefficients of $\varepsilon^k$**:
   - $O(\varepsilon^1)$: $x_1 + 1 = 0 \implies x_1 = -1$.
   - $O(\varepsilon^2)$: $x_2 + 5x_1 = 0 \implies x_2 = -5(-1) = 5$.

Thus:

$$
x(\varepsilon) = 1 - \varepsilon + 5\varepsilon^2 + O(\varepsilon^3)
$$

$$
\boxed{ x(\varepsilon) = 1 - \varepsilon + 5\varepsilon^2 + O(\varepsilon^3) }
$$

**Key takeaway**
Perturbation methods solve non-linear algebraic and differential equations via formal power series expansions.

In [35]:
# L3.4 — the perturbation series x(eps) = 1 - eps + 5 eps^2 against the true root.
from scipy.optimize import brentq

series = lambda e: 1 - e + 5 * e ** 2
print(f"{'eps':>10s} {'true root':>18s} {'1 - e + 5e^2':>18s} {'|difference|':>14s} {'35 eps^3':>12s}")
for eps in (1e-1, 1e-2, 1e-3, 1e-4):
    root = brentq(lambda x: x + eps * x ** 5 - 1, 0.0, 2.0, xtol=1e-15, rtol=8.9e-16)
    d = abs(root - series(eps))
    print(f"{eps:10.0e} {root:18.14f} {series(eps):18.14f} {d:14.3e} {35*eps**3:12.3e}")
    if eps <= 1e-2:
        assert d < 2 * 35 * eps ** 3
print("\nthe residual tracks 35 eps^3 exactly: the next coefficient is x_3 = -35, "
      "so 1 - eps + 5 eps^2 is correct through O(eps^2)")

       eps          true root       1 - e + 5e^2   |difference|     35 eps^3
     1e-01   0.93031373778061   0.95000000000000      1.969e-02    3.500e-02
     1e-02   0.99046761864356   0.99050000000000      3.238e-05    3.500e-05
     1e-03   0.99900496528249   0.99900500000000      3.472e-08    3.500e-08
     1e-04   0.99990004996503   0.99990005000000      3.497e-11    3.500e-11

the residual tracks 35 eps^3 exactly: the next coefficient is x_3 = -35, so 1 - eps + 5 eps^2 is correct through O(eps^2)


### Problem L3.5 — Generating Function of the Harmonic Numbers
**Source:** Polya & Szego, *Problems and Theorems in Analysis I*

**Statement**

Let $H_n = \sum_{k=1}^n \frac{1}{k}$ be the $n$-th harmonic number.  
(a) Prove that $\sum_{n=1}^\infty H_n x^n = \frac{-\ln(1-x)}{1-x}$ for $\lvert x \rvert \lt 1$.  
(b) Derive the asymptotic behavior of $H_n$ from the Cauchy product.

**Intuition**
Harmonic numbers are partial sums of $a_k = \frac{1}{k}$. Multiplying by $\frac{1}{1-x} = \sum x^n$ computes partial sums of power series coefficients.

**Solution**
(a) Consider $A(x) = -\ln(1-x) = \sum_{k=1}^\infty \frac{x^k}{k}$ and $B(x) = \frac{1}{1-x} = \sum_{m=0}^\infty x^m$.  
Compute Cauchy product $C(x) = A(x) B(x)$:

$$
C(x) = \sum_{n=1}^\infty c_n x^n \quad \text{where } c_n = \sum_{k=1}^n \left(\frac{1}{k}\right)(1) = H_n
$$

Thus:

$$
\sum_{n=1}^\infty H_n x^n = \frac{-\ln(1-x)}{1-x}
$$

(b) The asymptotics of $H_n$ are obtained directly, by comparing the sum to an integral rather
than by reading them off the generating function. Since $1/k$ is decreasing,

$$
\int_k^{k+1} \frac{dx}{x} \le \frac{1}{k} \le \int_{k-1}^{k} \frac{dx}{x} \quad (k \ge 2),
$$

so summing for $k = 1, \dots, n$ gives $\ln(n+1) \le H_n \le 1 + \ln n$. Hence $H_n - \ln n$ is
bounded; writing $\gamma_n = H_n - \ln n$, the bracketing above shows $\gamma_n$ is decreasing and
bounded below by $0$, so it converges to a limit $\gamma$ (the Euler–Mascheroni constant,
$\gamma \approx 0.5772$). This gives $H_n = \ln n + \gamma + o(1)$ as $n \to \infty$.

$$
\boxed{ \sum_{n=1}^\infty H_n x^n = \frac{-\ln(1-x)}{1-x}, \qquad H_n = \ln n + \gamma + o(1) }
$$

**Key takeaway**
Multiplying a generating function by $\frac{1}{1-x}$ transforms its coefficient sequence into its running partial sums.

In [36]:
# L3.5 — the generating function of H_n, and H_n = ln n + gamma + o(1).
H = np.concatenate(([0.0], np.cumsum(1.0 / np.arange(1, 3001))))
print(f"{'x':>6s} {'sum H_n x^n':>20s} {'-ln(1-x)/(1-x)':>20s} {'|difference|':>14s}")
for x in (0.2, 0.5, 0.8):
    s = float(np.sum(H[1:1500] * x ** np.arange(1, 1500)))
    closed = -np.log(1 - x) / (1 - x)
    print(f"{x:6.2f} {s:20.13f} {closed:20.13f} {abs(s-closed):14.3e}")
    assert abs(s - closed) < 1e-10
gamma = 0.5772156649015329
print(f"\n{'n':>8s} {'H_n':>16s} {'ln n + gamma':>16s} {'difference':>14s} {'1/(2n)':>12s}")
for n in (10, 100, 1000, 3000):
    print(f"{n:8d} {H[n]:16.12f} {np.log(n)+gamma:16.12f} {H[n]-np.log(n)-gamma:14.3e}"
          f" {1/(2*n):12.3e}")
    assert abs(H[n] - np.log(n) - gamma - 1 / (2 * n)) < 0.2 / n ** 2

     x          sum H_n x^n       -ln(1-x)/(1-x)   |difference|
  0.20      0.2789294391428      0.2789294391428      5.551e-17
  0.50      1.3862943611199      1.3862943611199      0.000e+00
  0.80      8.0471895621705      8.0471895621705      0.000e+00

       n              H_n     ln n + gamma     difference       1/(2n)
      10   2.928968253968   2.879800757896      4.917e-02    5.000e-02
     100   5.187377517640   5.182385850890      4.992e-03    5.000e-03
    1000   7.485470860550   7.484970943884      4.999e-04    5.000e-04
    3000   8.583749889959   8.583583232552      1.667e-04    1.667e-04


### Problem L3.6 — A Smooth Function with Zero Radius of Convergence
**Source:** Kaczor & Nowak, *Problems in Mathematical Analysis III*

**Statement**

Define $f(x) = \sum_{n=1}^\infty e^{-n} \cos(n^2 x)$.  
(a) Prove that $f \in C^\infty(\mathbb{R})$.  
(b) Show that the radius of convergence of its Maclaurin series centered at $x=0$ is $R = 0$.

**Intuition**
$e^{-n}$ decays so fast that term-by-term differentiation of any order $k$ converges uniformly. However, the derivative at 0 grows as $(n^2)^k$, causing coefficient growth faster than $k!$.

**Solution**
(a) **$C^\infty$ Smoothness**:  
The $k$-th derivative of the general term is $u_n^{(k)}(x) = \pm e^{-n} n^{2k} \cos(n^2 x)$ or $\pm e^{-n} n^{2k} \sin(n^2 x)$.  
Bound: $\lvert u_n^{(k)}(x) \rvert \le n^{2k} e^{-n}$.  
Since $\sum_{n=1}^\infty n^{2k} e^{-n}$ converges by the ratio test for every fixed $k \ge 0$, by the Weierstrass M-Test, the series for $f^{(k)}(x)$ converges uniformly on $\mathbb{R}$. Thus $f \in C^\infty(\mathbb{R})$.

(b) **Maclaurin Coefficient Growth**:  
Compute $f^{(2k)}(0)$:

$$
\lvert f^{(2k)}(0) \rvert = \sum_{n=1}^\infty e^{-n} n^{4k} \gt e^{-m} m^{4k} \quad \text{for any single term } m
$$

Set $m = 4k$:

$$
\lvert f^{(2k)}(0) \rvert \gt e^{-4k} (4k)^{4k}
$$

The Maclaurin coefficient $a_{2k} = \frac{f^{(2k)}(0)}{(2k)!}$:

$$
\lvert a_{2k} \rvert \gt \frac{e^{-4k} (4k)^{4k}}{(2k)!}
$$

Using Stirling's approximation $(2k)! \sim \sqrt{4\pi k} (2k/e)^{2k}$:

Since $(4k)^{4k} = 2^{4k}(2k)^{4k}$, substituting Stirling gives

$$
\lvert a_{2k} \rvert \gt \frac{e^{-4k} \, 2^{4k} \, (2k)^{4k}}{\sqrt{4\pi k}\,(2k/e)^{2k}}
= \frac{2^{4k} (2k)^{2k}}{e^{2k}\sqrt{4\pi k}} .
$$

Take the $2k$-th root. Since $(\sqrt{4\pi k})^{1/(2k)} \to 1$,

$$
\limsup_{k\to\infty} \lvert a_{2k} \rvert^{\frac{1}{2k}} \ge \lim_{k\to\infty} \frac{2^{2} \cdot 2k}{e} = \lim_{k\to\infty} \frac{8k}{e} = \infty .
$$

Thus $R = 1/\infty = 0$.

$$
\boxed{ f \in C^\infty(\mathbb{R}), \text{ but Maclaurin series has } R = 0 }
$$

**Key takeaway**
Uniform convergence guarantees infinite differentiability, but derivative growth can destroy power series convergence.

In [37]:
# L3.6 — f(x) = sum e^{-n} cos(n^2 x) is smooth, but its Maclaurin coefficients explode.
ns = np.arange(1, 400)
w = np.exp(-ns.astype(float))
print("term bounds for the k-th differentiated series, sum n^{2k} e^{-n}:")
for k in (0, 2, 5, 10, 20):
    tot = float(np.sum(ns.astype(float) ** (2 * k) * w))
    print(f"  k = {k:2d}:  sum n^(2k) e^-n = {tot:.6e}   (finite -> uniform convergence)")
    assert np.isfinite(tot)
print("\nMaclaurin coefficients a_(2k) = (-1)^k sum n^{4k} e^{-n} / (2k)! :")
from math import lgamma
print(f"{'k':>4s} {'log10 |a_2k|':>16s} {'|a_2k|^(1/(2k))':>18s}")
for k in (2, 4, 8, 16, 32):
    log_num = np.log(np.sum(ns.astype(float) ** (4 * k) * w))
    log_a = log_num - lgamma(2 * k + 1)
    print(f"{k:4d} {log_a/np.log(10):16.4f} {np.exp(log_a/(2*k)):18.4e}")
prev = None
for k in (2, 4, 8, 16, 32):
    log_a = np.log(np.sum(ns.astype(float) ** (4 * k) * w)) - lgamma(2 * k + 1)
    cur = np.exp(log_a / (2 * k))
    if prev is not None:
        assert cur > prev
    prev = cur
print("\n|a_2k|^(1/(2k)) grows without bound, so limsup = infinity and R = 0")

term bounds for the k-th differentiated series, sum n^{2k} e^{-n}:
  k =  0:  sum n^(2k) e^-n = 5.819767e-01   (finite -> uniform convergence)
  k =  2:  sum n^(2k) e^-n = 2.400333e+01   (finite -> uniform convergence)
  k =  5:  sum n^(2k) e^-n = 3.628800e+06   (finite -> uniform convergence)
  k = 10:  sum n^(2k) e^-n = 2.432902e+18   (finite -> uniform convergence)
  k = 20:  sum n^(2k) e^-n = 8.159153e+47   (finite -> uniform convergence)

Maclaurin coefficients a_(2k) = (-1)^k sum n^{4k} e^{-n} / (2k)! :
   k     log10 |a_2k|    |a_2k|^(1/(2k))
   2           3.2253         6.4022e+00
   4           8.7151         1.2285e+01
   8          22.0996         2.4056e+01
  16          53.6832         4.7599e+01
  32              inf                inf

|a_2k|^(1/(2k)) grows without bound, so limsup = infinity and R = 0


/tmp/ipykernel_251586/1919406641.py:13: RuntimeWarning: overflow encountered in power
  log_num = np.log(np.sum(ns.astype(float) ** (4 * k) * w))
/tmp/ipykernel_251586/1919406641.py:18: RuntimeWarning: overflow encountered in power
  log_a = np.log(np.sum(ns.astype(float) ** (4 * k) * w)) - lgamma(2 * k + 1)


### Problem L3.7 — Euler's Reflection Formula for the Dilogarithm
**Source:** L. Lewin, *Polylogarithms and Associated Functions* (1981), Ch. 1; the reflection formula is due to Euler.

**Statement**

The dilogarithm function is defined as $\operatorname{Li}_2(x) = \sum_{n=1}^\infty \frac{x^n}{n^2}$ for $\lvert x \rvert \le 1$.  
Prove the identity $\operatorname{Li}_2(x) + \operatorname{Li}_2(1-x) = \frac{\pi^2}{6} - \ln(x)\ln(1-x)$ for $x \in (0, 1)$.

**Intuition**
Differentiate both sides with respect to $x$. Since $\frac{d}{dx}\operatorname{Li}_2(x) = -\frac{\ln(1-x)}{x}$, derivative matching reduces the identity to a constant, evaluated at $x = \frac{1}{2}$.

**Solution**
1. **Differentiate $\operatorname{Li}_2(x)$**:

$$
   \frac{d}{dx} \operatorname{Li}_2(x) = \frac{d}{dx} \sum_{n=1}^\infty \frac{x^n}{n^2} = \sum_{n=1}^\infty \frac{x^{n-1}}{n} = \frac{1}{x} \sum_{n=1}^\infty \frac{x^n}{n} = -\frac{\ln(1-x)}{x}
$$

2. **Differentiate Left Hand Side $f(x) = \operatorname{Li}_2(x) + \operatorname{Li}_2(1-x)$** (chain rule on the second term, with $u = 1-x$, $du/dx = -1$):

$$
   f'(x) = -\frac{\ln(1-x)}{x} + \left( -\frac{\ln(1-(1-x))}{1-x} \right)(-1) = -\frac{\ln(1-x)}{x} + \frac{\ln(x)}{1-x}
$$

3. **Differentiate Right Hand Side $g(x) = \frac{\pi^2}{6} - \ln(x)\ln(1-x)$** (product rule):

$$
   g'(x) = -\left( \frac{1}{x}\ln(1-x) + \ln(x) \cdot \frac{-1}{1-x} \right) = -\frac{\ln(1-x)}{x} + \frac{\ln x}{1-x}
$$

Since $f'(x) = g'(x)$, $f(x) - g(x) = C$ (constant).

4. **Evaluate Constant via Limit $x \to 0^+$**:
   $\operatorname{Li}_2(0) = 0$, $\operatorname{Li}_2(1) = \sum_{n=1}^\infty \frac{1}{n^2} = \frac{\pi^2}{6}$.  
   $\lim_{x\to 0^+} \ln(x)\ln(1-x) = \lim_{x\to 0^+} \ln(x)(-x + O(x^2)) = 0$.  
   Thus $C = 0$.

$$
\boxed{ \operatorname{Li}_2(x) + \operatorname{Li}_2(1-x) = \frac{\pi^2}{6} - \ln(x)\ln(1-x) }
$$

**Key takeaway**
Power series derivative identities prove functional equations for special transcendental functions.

In [38]:
# L3.7 — Euler's reflection formula for the dilogarithm.
def li2(x, N=200000):
    n = np.arange(1, N + 1)
    return float(np.sum(x ** n / n ** 2))

print(f"{'x':>6s} {'Li2(x)+Li2(1-x)':>20s} {'pi^2/6 - ln x ln(1-x)':>24s} {'|difference|':>14s}")
for x in (0.1, 0.25, 0.5, 0.75, 0.9):
    lhs = li2(x) + li2(1 - x)
    rhs = np.pi ** 2 / 6 - np.log(x) * np.log(1 - x)
    print(f"{x:6.2f} {lhs:20.12f} {rhs:24.12f} {abs(lhs-rhs):14.3e}")
    assert abs(lhs - rhs) < 1e-8
print(f"\nLi2(1) = {li2(1.0):.10f}   pi^2/6 = {np.pi**2/6:.10f}")
print(f"Li2(1/2) = {li2(0.5):.10f}   (pi^2/12 - (ln 2)^2 / 2) = "
      f"{np.pi**2/12 - np.log(2)**2/2:.10f}")
assert abs(li2(0.5) - (np.pi ** 2 / 12 - np.log(2) ** 2 / 2)) < 1e-8

     x      Li2(x)+Li2(1-x)    pi^2/6 - ln x ln(1-x)   |difference|
  0.10       1.402332514104           1.402332514104      2.220e-16
  0.25       1.246122032013           1.246122032013      2.220e-16


  0.50       1.164481052930           1.164481052930      0.000e+00


  0.75       1.246122032013           1.246122032013      2.220e-16
  0.90       1.402332514104           1.402332514104      2.220e-16

Li2(1) = 1.6449290669   pi^2/6 = 1.6449340668
Li2(1/2) = 0.5822405265   (pi^2/12 - (ln 2)^2 / 2) = 0.5822405265


### Problem L3.8 — Laplace's Method for a Gaussian Integral
**Source:** Cambridge Mathematical Tripos, Part II

**Statement**

Determine the leading asymptotic behavior of the integral as $\lambda \to \infty$:

$$
I(\lambda) = \int_{-\infty}^\infty e^{-\lambda x^2} \cos(x) \, dx
$$

**Intuition**
For large $\lambda$, $e^{-\lambda x^2}$ is sharply localized around $x=0$. Taylor expand $\cos(x) = 1 - \frac{x^2}{2} + \dots$ around $x=0$ and evaluate Gaussian integrals.

**Solution**
1. **Taylor expand $\cos(x)$**:

$$
   \cos(x) = 1 - \frac{x^2}{2} + O(x^4)
$$

2. **Substitute into Integral**:

$$
   I(\lambda) = \int_{-\infty}^\infty e^{-\lambda x^2} \left( 1 - \frac{x^2}{2} + O(x^4) \right) dx
$$

3. **Evaluate Standard Gaussian Integrals**:
   - $\int_{-\infty}^\infty e^{-\lambda x^2} dx = \sqrt{\frac{\pi}{\lambda}}$
   - $\int_{-\infty}^\infty x^2 e^{-\lambda x^2} dx = \frac{\sqrt{\pi}}{2 \lambda^{\frac{3}{2}}}$

4. **Combine Terms**:

$$
   I(\lambda) = \sqrt{\frac{\pi}{\lambda}} - \frac{1}{2} \left( \frac{\sqrt{\pi}}{2 \lambda^{\frac{3}{2}}} \right) + O(\lambda^{-\frac{5}{2}}) = \sqrt{\frac{\pi}{\lambda}} \left( 1 - \frac{1}{4\lambda} + O(\lambda^{-2}) \right)
$$

$$
\boxed{ I(\lambda) = \sqrt{\frac{\pi}{\lambda}} \left( 1 - \frac{1}{4\lambda} + O(\lambda^{-2}) \right) }
$$

**Key takeaway**
Laplace's method replaces global integrals with local Taylor series around peak points.

In [39]:
# L3.8 — Laplace's method against the exact Gaussian-cosine integral.
from scipy.integrate import quad

exact = lambda lam: np.sqrt(np.pi / lam) * np.exp(-1 / (4 * lam))
approx = lambda lam: np.sqrt(np.pi / lam) * (1 - 1 / (4 * lam))
print(f"{'lambda':>8s} {'quadrature':>18s} {'exact closed form':>20s} {'two-term':>18s}"
      f" {'|rel. err|':>12s}")
for lam in (2.0, 5.0, 20.0, 100.0):
    num = quad(lambda x: np.exp(-lam * x ** 2) * np.cos(x), -np.inf, np.inf)[0]
    print(f"{lam:8.1f} {num:18.12f} {exact(lam):20.12f} {approx(lam):18.12f}"
          f" {abs(approx(lam)-num)/num:12.3e}")
    assert abs(num - exact(lam)) < 1e-10
    assert abs(approx(lam) - num) / num < 1.0 / lam ** 2
print("\nthe two-term Laplace result has relative error O(1/lambda^2), as claimed")

  lambda         quadrature    exact closed form           two-term   |rel. err|
     2.0     1.106045844146       1.106045844146     1.096649870151    8.495e-03
     5.0     0.754006708882       0.754006708882     0.753032186545    1.292e-03
    20.0     0.391409405521       0.391409405521     0.391378570639    7.878e-05
   100.0     0.176802825058       0.176802825058     0.176802271628    3.130e-06

the two-term Laplace result has relative error O(1/lambda^2), as claimed


### Problem L3.9 — Stirling's Formula by Euler-Maclaurin
**Source:** Cambridge Mathematical Tripos / Advanced Analysis

**Statement**

Derive the leading terms of Stirling's approximation $\ln(n!) = n\ln n - n + \frac{1}{2}\ln(2\pi n) + O(\frac{1}{n})$ using the Euler-Maclaurin expansion of $\sum_{k=1}^n \ln k$.

**Intuition**
Express $\ln(n!) = \sum_{k=1}^n \ln k$. Compare the sum to the integral $\int_1^n \ln x \, dx = n\ln n - n + 1$, using trapezoidal Taylor approximations.

**Solution**
1. **Integral Approximation**:

$$
   \int_1^n \ln x \, dx = [x\ln x - x]_1^n = n\ln n - n + 1
$$

2. **Trapezoidal Rule Difference**:  
   By the Euler-Maclaurin summation formula:

$$
   \sum_{k=1}^n \ln k = \int_1^n \ln x \, dx + \frac{1}{2}(\ln 1 + \ln n) + \int_1^n \frac{x - \lfloor x \rfloor - \frac{1}{2}}{x} dx
$$

3. **Evaluate Constant**:  
   The remaining integral converges as $n \to \infty$ to a constant $C = 1 + \int_1^\infty \frac{x - \lfloor x \rfloor - \frac{1}{2}}{x} dx$.  
   Using Wallis' product formula identifies $C = \frac{1}{2}\ln(2\pi)$.

4. **Combine Terms**:

$$
   \ln(n!) = n\ln n - n + \frac{1}{2}\ln n + \frac{1}{2}\ln(2\pi) + O(\frac{1}{n})
$$

$$
   \ln(n!) = n\ln n - n + \frac{1}{2}\ln(2\pi n) + O(\frac{1}{n})
$$

$$
\boxed{ \ln(n!) = n\ln n - n + \frac{1}{2}\ln(2\pi n) + O(\frac{1}{n}) }
$$

**Key takeaway**
Euler-Maclaurin formula leverages local Taylor remainder terms to convert discrete sums into continuous asymptotic integrals.

In [40]:
# L3.9 — Stirling's formula ln(n!) = n ln n - n + (1/2) ln(2 pi n) + O(1/n).
from math import lgamma

print(f"{'n':>8s} {'ln(n!)':>20s} {'Stirling':>20s} {'difference':>14s} {'1/(12n)':>12s}")
for n in (5, 10, 50, 200, 1000):
    exact = lgamma(n + 1)
    stir = n * np.log(n) - n + 0.5 * np.log(2 * np.pi * n)
    print(f"{n:8d} {exact:20.12f} {stir:20.12f} {exact-stir:14.3e} {1/(12*n):12.3e}")
    assert abs((exact - stir) - 1 / (12 * n)) < 1e-3 / n
print("\nthe residual matches the next Euler-Maclaurin term 1/(12n) to three digits,"
      "\nconfirming both the n ln n - n + (1/2) ln(2 pi n) main terms and the O(1/n) claim")

       n               ln(n!)             Stirling     difference      1/(12n)
       5       4.787491742782       4.770847051592      1.664e-02    1.667e-02
      10      15.104412573076      15.096082009642      8.331e-03    8.333e-03
      50     148.477766951773     148.476100307326      1.667e-03    1.667e-03
     200     863.231987192405     863.231570526086      4.167e-04    4.167e-04
    1000    5912.128178488163    5912.128095154832      8.333e-05    8.333e-05

the residual matches the next Euler-Maclaurin term 1/(12n) to three digits,
confirming both the n ln n - n + (1/2) ln(2 pi n) main terms and the O(1/n) claim


### Problem L3.10 — Fourier Series of the Sawtooth via the Complex Logarithm
**Source:** Classical Analysis / Master Tripos

**Statement**

Prove that for all $\theta \in (0, 2\pi)$:

$$
\sum_{n=1}^\infty \frac{\sin(n\theta)}{n} = \frac{\pi - \theta}{2}
$$

using the principal branch of the complex logarithm $\operatorname{Log}(1 - z)$ and Abel's Theorem.

**Intuition**
Consider the power series $\sum_{n=1}^\infty \frac{z^n}{n} = -\operatorname{Log}(1-z)$ for $z = e^{i\theta}$. Taking the imaginary part yields $\sum \frac{\sin(n\theta)}{n}$.

**Solution**
1. **Complex Power Series**:  
   For $\lvert z \rvert \lt 1$:

$$
   f(z) = \sum_{n=1}^\infty \frac{z^n}{n} = -\operatorname{Log}(1-z)
$$

2. **Evaluate on Boundary $z = e^{i\theta}$ ($\theta \in (0, 2\pi)$)**:

$$
   1 - e^{i\theta} = 1 - \cos\theta - i\sin\theta = 2\sin^2(\theta/2) - 2i\sin\left(\frac{\theta}{2}\right)\cos\left(\frac{\theta}{2}\right)
$$

   Factor out $-2i\sin\left(\frac{\theta}{2}\right) = 2\sin\left(\frac{\theta}{2}\right) e^{-i\frac{\pi}{2}}$:

$$
   1 - e^{i\theta} = 2\sin\left(\frac{\theta}{2}\right) \left( \sin\left(\frac{\theta}{2}\right) - i\cos\left(\frac{\theta}{2}\right) \right) = 2\sin\left(\frac{\theta}{2}\right) e^{i(\theta/2 - \frac{\pi}{2})}
$$

3. **Take Principal Complex Logarithm**:

$$
   -\operatorname{Log}(1 - e^{i\theta}) = -\ln\left( 2\sin\left(\frac{\theta}{2}\right) \right) - i \left( \frac{\theta - \pi}{2} \right) = -\ln\left( 2\sin\left(\frac{\theta}{2}\right) \right) + i \left( \frac{\pi - \theta}{2} \right)
$$

4. **Extract Imaginary Part**:

$$
   \operatorname{Im} \left( \sum_{n=1}^\infty \frac{e^{in\theta}}{n} \right) = \sum_{n=1}^\infty \frac{\sin(n\theta)}{n} = \frac{\pi - \theta}{2}
$$

5. **Rigor via Abel's Theorem**:  
   Since $\sum \frac{\sin(n\theta)}{n}$ converges for all $\theta \in (0, 2\pi)$ by Dirichlet's test, Abel's theorem guarantees that taking $\lim_{r \to 1^-} \operatorname{Im}(-\operatorname{Log}(1 - r e^{i\theta}))$ equals the boundary sum.

$$
\boxed{ \sum_{n=1}^\infty \frac{\sin(n\theta)}{n} = \frac{\pi - \theta}{2} \quad \text{for } \theta \in (0, 2\pi) }
$$

**Key takeaway**
Complex power series analytical continuation seamlessly unifies power series with Fourier series representations.

In [41]:
# L3.10 — the sawtooth Fourier series sum sin(n theta)/n = (pi - theta)/2 on (0, 2 pi).
N = 400000
n = np.arange(1, N + 1)
print(f"{'theta':>8s} {'partial sum (4e5)':>20s} {'(pi - theta)/2':>18s} {'|difference|':>14s}")
for theta in (0.4, 1.0, np.pi, 4.0, 6.0):
    s = float(np.sum(np.sin(n * theta) / n))
    closed = (np.pi - theta) / 2
    print(f"{theta:8.4f} {s:20.12f} {closed:18.12f} {abs(s-closed):14.3e}")
    assert abs(s - closed) < 1e-4
# the Abel route: the interior limit r -> 1^-
print("\nAbel limit of Im(-Log(1 - r e^{i theta})) as r -> 1^-, at theta = 1.0:")
for r in (0.9, 0.99, 0.999, 0.9999):
    val = float(np.imag(-np.log(1 - r * np.exp(1j * 1.0))))
    print(f"  r = {r:.4f}: {val:.12f}   target {(np.pi - 1.0)/2:.12f}")
assert abs(float(np.imag(-np.log(1 - 0.9999 * np.exp(1j)))) - (np.pi - 1.0) / 2) < 1e-4

   theta    partial sum (4e5)     (pi - theta)/2   |difference|
  0.4000       1.370793550787     1.370796326795      2.776e-06


  1.0000       1.070793883876     1.070796326795      2.443e-06


  3.1416      -0.000000000000     0.000000000000      7.491e-15
  4.0000      -0.429203868978    -0.429203673205      1.958e-07
  6.0000      -1.429198885417    -1.429203673205      4.788e-06

Abel limit of Im(-Log(1 - r e^{i theta})) as r -> 1^-, at theta = 1.0:
  r = 0.9000: 0.974751288822   target 1.070796326795
  r = 0.9900: 1.061598155449   target 1.070796326795
  r = 0.9990: 1.069880625339   target 1.070796326795
  r = 0.9999: 1.070704797833   target 1.070796326795
